<div align="center">

# Hafta 9: Model güvenilirliği ve açıklanabilir yapay zekâ

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/lab.html) · [PDF ders notu](Hafta09_Ders_Notlari.pdf) · [Colab'da aç](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-09/NB09_guvenilirlik_ve_aciklanabilir_yapay_zeka.ipynb)

</div>

**Bu defter iki kısımdan oluşur.** Birincisinde kodu bir yapay zekâ (YZ) asistanı yazar; asistana ne yapacağını yazılı bir talimat, yani bir prompt (istem) anlatır. İkincisinde kodu Python ile kendiniz yazarsınız. İki kısım da aynı verilerle aynı sonuçlara varır; ders sayfasındaki her uygulama durağında bunlardan birini seçin.

| | Kimler için? | Ne yapılır? |
|---|---|---|
| **[Kısım 1: Prompt yolu](#part-1)** | Kod yazmayan katılımcılar | Kodu, hazır promptlarla bir YZ asistanı yazar; siz çalıştırır ve sonucu denetlersiniz. |
| **[Kısım 2: Python yolu](#part-2)** | Python öğrenmek isteyen katılımcılar | Kodu alıştırmalarla adım adım kendiniz yazar ve çalıştırırsınız. |

**Nerede çalışır?** Aynı defter, hiçbir kurulum gerektirmeyen Google Colab'da da kendi bilgisayarınızdaki Jupyter'de de (Anaconda; ders klasöründeki SETUP.tr.md dosyasına bakın) çalışır. Hangisini tercih ederseniz onu kullanın. Colab'da önce *Dosya → Drive'a kopya kaydet* (*File → Save a copy in Drive*) komutuyla kendi kopyanızı kaydedin; aksi hâlde sekme kapanınca yaptığınız değişiklikler kaybolur. 3. bölümdeki kesin Shapley değerleri yaklaşık yarım dakika sürer; 5. bölümdeki eko ağı da sıradan bir dizüstü bilgisayarda bir dakikadan kısa sürede eğitilir. Ekran kartı gerekmez. Hangi kısmı seçerseniz seçin önce 0. bölümü çalıştırın. Bu bölüm ders dosyalarını ve paketleri hazırlar.

**Veri ve mahremiyet.** Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelmektedir. Gerçek hasta verisi yalnızca kendi bilgisayarınızda, kurumunuzun kurallarına uygun olarak ve etik kurul onayıyla işlenir; bulut tabanlı bir deftere ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

İlk hücre ders klasörünü bulur. Defter tek başına açıldıysa, örneğin Google Colab'da, `vclinic` ders paketini ve veri dosyalarını GitHub'dan indirir. İkinci hücre paketleri yükler, mesajların dilini seçer ve haftanın isteğe bağlı paketlerini denetler. İkinci hücre 4. haftada kullanılan 300 CAMUS hastasını da yaklaşık 250 MB olarak bir ile üç dakikada yeniden indirir; zaten bulunan dosyalar atlanır. Cleveland verisi ve göğüs ağrısı ünitesi ders paketiyle birlikte gelir.

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
    "data/uci-heart-disease/processed.cleveland.data",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopyada, örneğin Colab'da, paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy
import itertools
import math

from scipy.stats import spearmanr
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic import camus, triage
from vclinic.datasets import load_cleveland

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError as error:
    raise ImportError("PyTorch kurulu değil. SETUP.tr.md dosyasında anlatıldığı gibi kurun, ardından çekirdeği yeniden başlatın.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 9, 0):
    raise ImportError("vclinic ders paketi bu defterden daha eskidir. Ders klasörünü güncelleyin (git pull komutuyla ya da ZIP dosyasını yeniden indirerek) ve çekirdeği yeniden başlatın.")
teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
camus.download()
print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
for name in ["shap"]:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        print(f"{name}: kurulu değil (isteğe bağlı; defter yerleşik bir yedek yol kullanır)")

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 9.1": (56, 0),
    "Alıştırma 9.2": (0.105, 0.001),
    "Alıştırma 9.3": (0.00357, 1e-05),
    "Alıştırma A": (33, 0),
    "Alıştırma B": (0.749, 0.002),
    "Alıştırma C": (56, 3),
    "Alıştırma D": (0.031, 0.02),
    "Alıştırma E": (0.824, 0.06),
    "Alıştırma F": (0.67, 0.08),
    "Alıştırma G": (2, 0),
    "Alıştırma H": (0.687, 0.01),
    "Alıştırma I": (723, 0),
    "Alıştırma J": (0.932, 0.02),
    "Adım 1.1": (0.904, 0.03),
    "Adım 1.2": (0.155, 0.03),
    "Adım 1.3": (-1.175, 0.01),
    "Adım 1.4": (0.138, 0.03),
    "Adım 2.1": (0.145, 0.03),
    "Adım 2.2": (0.7, 0.25),
    "Adım 2.3": (0.796, 0.06),
    "Adım 2.4": (0.317, 0.06),
    "Adım 3.1": (0.988, 0.02),
    "Adım 3.2": (253, 0),
    "Adım 3.3": (0.956, 0.001),
    "Adım 3.4": (0.417, 0.02),
})
print("Beklenen cevaplar hazır.")

In [ ]:
#@title Alıştırma promptlarının tabloları (bu hücreyi çalıştırın; Python yolunda okunması gerekmez)
def alistirma_tablosu(anahtar):
    """Bir alıştırma promptunun "veri" tablosu; haftanın verilerinden kurulur."""
    g = globals()
    k = anahtar
    need = lambda *names: [g[n] for n in names] if all(n in g for n in names) else (_ for _ in ()).throw(
        RuntimeError("Önce bu alıştırmadan önceki bölümleri çalıştırın."))
    sets = {"training": "eğitim", "validation": "doğrulama", "testing": "test", "test": "test"}
    if k == "9.2":
        rows = [(0, 0, 0, 0.552), (1, 0, 0, 0.660), (0, 1, 0, 0.617), (0, 0, 1, 0.456), (1, 1, 0, 0.799), (1, 0, 1, 0.584), (0, 1, 1, 0.533), (1, 1, 1, 0.727)]
        return pd.DataFrame(rows, columns=["göğüs ağrısı tipi", "egzersiz anjinası", "cinsiyet", "risk"])
    if k == "A":
        forest, X_test, y_test = need("forest", "X_test", "y_test")
        return pd.DataFrame({"hasta": X_test.index, "olasılık": forest.predict_proba(X_test)[:, 1].round(3),
                             "hastalık": y_test.to_numpy()})
    if k in ("B", "C", "D"):
        heart, X_train, X_test = need("heart", "X_train", "X_test")
        h = heart.loc[list(X_train.index) + list(X_test.index)]
        cp = {1.0: "tipik anjina", 2.0: "atipik anjina", 3.0: "anjina dışı ağrı", 4.0: "ağrı yok"}
        return pd.DataFrame({"hasta": h.index, "küme": np.where(h.index.isin(X_train.index), "eğitim", "test"),
                             "yaş": h["age"].astype(int), "cinsiyet": h["sex"].map({1.0: "erkek", 0.0: "kadın"}),
                             "istirahat kan basıncı": h["trestbps"].astype(int), "kolesterol": h["chol"].astype(int),
                             "en yüksek kalp hızı": h["thalach"].astype(int), "egzersiz anjinası": h["exang"].astype(int),
                             "ST çökmesi": h["oldpeak"], "göğüs ağrısı tipi": h["cp"].map(cp),
                             "hastalık": h["disease"]}).reset_index(drop=True)
    if k in ("E", "F", "G"):
        views, echo_x, outline, echo_y, echo_set = need("views", "echo_x", "outline", "echo_y", "echo_set")
        q = camus.table().set_index("patient")
        return pd.DataFrame({"hasta": [p for p, v in views], "görüntü": [v for p, v in views], "küme": [sets[s] for s in echo_set],
                             "görüntü kalitesi": [q.loc[p, "quality_2ch" if v == "2CH" else "quality_4ch"] for p, v in views],
                             "diyastol sonu": list(echo_x[:, 0]), "sistol sonu": list(echo_x[:, 1]),
                             "sınır": list(outline.astype(np.uint8)), "düşük ejeksiyon fraksiyonu": echo_y})
    if k == "H":
        A, B = need("A", "B")
        rows = []
        for name, d in (("A", A), ("B", B)):
            url = d["troponin_url"]
            rows.append(pd.DataFrame({"hastane": name, "küme": d["set"].map(sets).to_numpy() if name == "A" else "B hastanesi",
                                      "yaş": d["age"], "cinsiyet": d["sex"].map({"M": "erkek", "F": "kadın"}),
                                      "sigara": d["smoker"], "diyabet": d["diabetes"], "hipertansiyon": d["hypertension"],
                                      "hiperlipidemi": d["hyperlipidaemia"], "obezite": d["obesity"],
                                      "aile öyküsü": d["family_history"], "bilinen koroner arter hastalığı": d["known_cad"],
                                      "kalp hızı": d["heart_rate"], "sistolik basınç": d["sbp"], "eGFR": d["egfr"],
                                      "gelişte troponin": (d["troponin_0h"] / url).round(2),
                                      "bir saat sonra troponin": (d["troponin_1h"] / url).round(2),
                                      "ekokardiyogram yapıldı": d["echo_done"], "ejeksiyon fraksiyonu": d["lvef"],
                                      "duvar hareket bozukluğu": d["rwma"], "triyaj notu": d["note_tr"]}))
        return pd.concat(rows, ignore_index=True)
    if k == "I":
        B, pB = need("B", "pB")
        return pd.DataFrame({"sıra": np.arange(1, len(B) + 1), "olasılık": pB, "MACE": B["mace30"].to_numpy()})
    if k == "J":
        A, pA, va, te = need("A", "pA", "va", "te")
        m = va | te
        return pd.DataFrame({"küme": np.where(va, "doğrulama", "test")[m], "cinsiyet": A["sex"].map({"M": "erkek", "F": "kadın"}).to_numpy()[m],
                             "yaş": A["age"].to_numpy()[m], "olasılık": pA[m], "MACE": A["mace30"].to_numpy()[m]})
    raise KeyError(k)
print("Alıştırma promptlarının tabloları hazır.")

<a id="part-1"></a>

# Kısım 1: Prompt yolu

**Kimler için?** Bu kısım kod yazmayan katılımcılar içindir. Burada hiç kod yazmazsınız. Kodu, hazır bir prompttan, yani yazılı bir talimattan yararlanarak bir yapay zekâ (YZ) asistanı yazar; siz kodu çalıştırır ve sonucu denetlersiniz.

**Neye ihtiyacınız var?** Python kodu yazabilen bir YZ asistanı yeterlidir. Google Colab'ın içinde Gemini bulunur; pencerenin üstündeki Gemini düğmesiyle açılır. Başka bir tarayıcı sekmesinde açılan ChatGPT, Claude, Copilot ya da Gemini de aynı işi görür.

**Bir adım nasıl ilerler?**

1. Adımın amacını okuyun.
2. Promptun altındaki veri hücresini çalıştırın. Hücre, promptun tarif ettiği "veri" tablosunu hazırlar ve tablonun ilk ve son satırlarını gösterir.
3. Promptu kopyalayın: Fareyle seçin ve Ctrl+C'ye (Mac'te Cmd+C) basın.
4. Promptu YZ asistanına yapıştırın ve gönderin.
5. Asistanın verdiği kodu kopyalayın, veri hücresinin altındaki hücreye `#` ile başlayan satırın yerine yapıştırın ve hücreyi ▶ düğmesiyle ya da Shift+Enter ile çalıştırın.
6. Hemen altındaki kontrol hücresini çalıştırın. Hücre *doğru ✓* ya da *henüz değil ✗* yazar.
7. *Henüz değil* yazarsa aşağıdaki onarım promptunu kodla birlikte asistana gönderin. Sonuç yine yanlışsa *Hazır kod* hücresini çalıştırın. Bu hücre her zaman doğru sonucu verir; böylece bir sonraki adıma geçebilirsiniz.

**"veri" ve "cevap".** 1. haftada olduğu gibi her adımın veri hücresi, promptun tarif ettiği "veri" tablosunu hazırlar. Asistan sonucu "cevap" adıyla saklar; kontrol hücresi de sonucu bu addan okur.

**Her durağın başında** o durağın hazırlık hücresini bir kez çalıştırın. Hücre durağın verilerini yükler; böylece bir durağa başka bir gün de başlanabilir.

**Onarım promptu** (çalışan ama kontrolden geçmeyen kod için):

> Aşağıdaki kod şu görev için yazıldı: [promptun Görev satırını yapıştırın]. Kod çalışıyor, ancak dersin kontrolü "henüz değil" yazıyor. Hatayı bul, iki cümleyle açıkla ve düzeltilmiş kodu ver. Sonuç yine "cevap" adıyla saklanmalı.  
>
> [kodu buraya yapıştırın]  

**Hata mesajı için prompt** (hata vererek duran kod için):

> Aşağıdaki kod bir Jupyter defterinde çalıştırıldığında altta verilen hata mesajını veriyor. Hatanın nedenini sade bir dille açıkla ve düzeltilmiş kodu ver.  
>
> [kodu buraya yapıştırın]  
>
> [hata mesajını buraya yapıştırın]  

Haftanın bütün promptları [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-09/PROMPTLAR.md) de yer alır; GitHub her promptun yanında bir kopyalama düğmesi gösterir.

**Hiç değişmeyen kural.** Promptlarda yalnızca dersin sentetik ya da açık verileri bulunur. Bir YZ asistanına gerçek hasta verisi asla yapıştırılmaz.

## Durak 1: Koroner arter hastalığı modelinin ardındaki klinik etkenler

*Ders: [1. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#stop-1)* · *Python yolunda aynı sonuç: Kısım 2, 1-3. bölümler*

Bu durak oturumun mini projesidir. 2. haftanın koroner arter hastalığı için kurduğu iki model, yani bir rastgele orman ve bir lojistik regresyon, Cleveland verisinin eğitim hastalarıyla öğrenir. Modellerin 59 yaşındaki bir kadın olan 172 numaralı hasta için verdiği tahminler, kesin Shapley değerleriyle hastanın sekiz klinik değişkeni arasında paylaştırılır; lojistik regresyon için kapalı bir form, log-odds ölçeğinde aynı değerleri verir. Son adım her test hastasının değerlerini genel bir açıklamaya dönüştürür.

**Durağın sonunda:** Ormanın test AUC değerini (0,904), ormanda 172 numaralı hastanın göğüs ağrısı tipinin Shapley değerini (0,155), lojistik regresyonda cinsiyetinin kapalı formdaki değerini (−1,175) ve test hastalarında göğüs ağrısı tipinin ortalama mutlak Shapley değerini (0,138) bulacaksınız.

In [ ]:
#@title 1. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
_heart = load_cleveland()
# 2. haftanın ayırması, kendi sırasıyla: Önce eğitim, sonra test hastaları
_cad_tr, _cad_te = train_test_split(_heart.index.to_numpy(), test_size=0.25, stratify=_heart["disease"], random_state=RANDOM_STATE)
_CP = {1.0: "tipik anjina", 2.0: "atipik anjina", 3.0: "anjina dışı ağrı", 4.0: "ağrı yok"}


def _cad_table(order="split"):
    """Cleveland verisindeki 303 hasta; kümeleri, sekiz klinik değişken ve hastalıkla birlikte; 2. haftanın ayırmasının ya da hasta numaralarının sırasıyla."""
    h = _heart.loc[list(_cad_tr) + list(_cad_te)] if order == "split" else _heart
    return pd.DataFrame({"hasta": h.index, "küme": np.where(h.index.isin(_cad_tr), "eğitim", "test"),
                         "yaş": h["age"].astype(int), "cinsiyet": h["sex"].map({1.0: "erkek", 0.0: "kadın"}),
                         "istirahat kan basıncı": h["trestbps"].astype(int), "kolesterol": h["chol"].astype(int),
                         "en yüksek kalp hızı": h["thalach"].astype(int), "egzersiz anjinası": h["exang"].astype(int),
                         "ST çökmesi": h["oldpeak"], "göğüs ağrısı tipi": h["cp"].map(_CP),
                         "hastalık": h["disease"]}).reset_index(drop=True)


print(len(_heart), "hasta Cleveland verisinde,", int(_heart["disease"].sum()), "hastada koroner arter hastalığı")
print("1. durak hazır.")

### Adım 1.1: 2. haftanın iki modeli

**Amaç.** Bu adımda 2. haftanın iki modeli eğitim hastalarında yeniden kurulur ve test hastalarında değerlendirilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kalite ve güvenlik kurulunun kullanılmadan önce inceleyebilmesi için önceki bir analizdeki koroner arter hastalığı modellerini yeniden kurmak istiyorum.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani koroner anjiyografiye gönderilmiş hastalardan 303 hasta bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını (0-302), "küme" 2. haftanın kümesini verir ("eğitim", 227 hasta; "test", 76 hasta). Klinik değişkenler şunlardır: "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), "istirahat kan basıncı" (mmHg), "kolesterol" (mg/dL), "en yüksek kalp hızı" (egzersiz testinde ulaşılan en yüksek kalp hızı, dakikada), "egzersiz anjinası" (egzersiz testi anjinaya yol açtıysa 1), "ST çökmesi" (egzersizde ST segmentinin istirahate göre çökmesi, mm) ve "göğüs ağrısı tipi" ("tipik anjina", "atipik anjina", "anjina dışı ağrı" ya da "ağrı yok"). "hastalık" anjiyogramda koroner arter hastalığı görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Tabloyu bu sırayla bir modelin sütunlarına dönüştür: Yaş, erkek için 1 ve kadın için 0 olan cinsiyet, istirahat kan basıncı, kolesterol, en yüksek kalp hızı, egzersiz anjinası, ST çökmesi ve tipik anjinayı referans alan, atipik anjina, anjina dışı ağrı ve ağrı yok için 0 ve 1 değerli üç sütun. Satırları tablonun sırasında tut. Eğitim hastalarında iki model kur: Sütunları eğitim hastalarıyla standartlaştırılmış ve en fazla 1.000 yinelemeli bir lojistik regresyon ile 300 ağaçlı bir rastgele orman. Sonucun tekrarlanabilmesi için rastgelelik tohumunu 2026 olarak sabitle. İki modelin test hastalarındaki ROC eğrisi altındaki alanını (AUC) ve her birinin 172 numaralı hastaya verdiği olasılığı hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki test AUC değerini ve 172 numaralı hastanın iki olasılığını yazdır; ormanın test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Hiçbir test hastası modellerden birini kurmak için kullanılmamalı ve AUC 0 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 1.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _cad_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.1 kontrolü
teach.check_step("Adım 1.1", "cevap", globals())

In [ ]:
#@title Adım 1.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _cad_table()
d = veri
agri = d["göğüs ağrısı tipi"]
X = pd.DataFrame({"yaş": d["yaş"], "erkek": (d["cinsiyet"] == "erkek").astype(int),
                    "istirahat kan basıncı": d["istirahat kan basıncı"], "kolesterol": d["kolesterol"],
                    "en yüksek kalp hızı": d["en yüksek kalp hızı"], "egzersiz anjinası": d["egzersiz anjinası"],
                    "ST çökmesi": d["ST çökmesi"],
                    "atipik anjina": (agri == "atipik anjina").astype(int),
                    "anjina dışı ağrı": (agri == "anjina dışı ağrı").astype(int),
                    "ağrı yok": (agri == "ağrı yok").astype(int)})
y = d["hastalık"].to_numpy()
eg_ = (d["küme"] == "eğitim").to_numpy()
te_ = ~eg_
oyuncular = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

orman = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X[eg_], y[eg_])
lojistik = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[eg_], y[eg_])

vaka = d["hasta"] == 172
print("test AUC: orman", round(roc_auc_score(y[te_], orman.predict_proba(X[te_])[:, 1]), 3),
      "| lojistik regresyon", round(roc_auc_score(y[te_], lojistik.predict_proba(X[te_])[:, 1]), 3))
print("172 numaralı hasta: orman", round(orman.predict_proba(X[vaka])[0, 1], 3),
      "| lojistik regresyon", round(lojistik.predict_proba(X[vaka])[0, 1], 3))
cevap = round(float(roc_auc_score(y[te_], orman.predict_proba(X[te_])[:, 1])), 3)
teach.check_step("Adım 1.1", "cevap", globals())

**Sonuç ne gösteriyor?** Orman 0,904 test AUC değerine ulaşır. İki model de hastaları iyi sıralar; 172 numaralı hastaya ise oldukça farklı olasılıklar verir ve sonraki adımlar bu farkı açıklar.

### Adım 1.2: 172 numaralı hastanın Shapley değerleri

**Amaç.** Bu adımda ormanın 172 numaralı hastaya verdiği olasılık, hastanın sekiz klinik değişkeni arasında paylaştırılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kurul, ormanın egzersiz testinde anjinası ortaya çıkan 59 yaşındaki bir kadın olan 172 numaralı hastaya, ortalama hastaya kıyasla neden yüksek bir koroner arter hastalığı olasılığı verdiğini soruyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani koroner anjiyografiye gönderilmiş hastalardan 303 hasta bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını (0-302), "küme" 2. haftanın kümesini verir ("eğitim", 227 hasta; "test", 76 hasta). Klinik değişkenler şunlardır: "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), "istirahat kan basıncı" (mmHg), "kolesterol" (mg/dL), "en yüksek kalp hızı" (egzersiz testinde ulaşılan en yüksek kalp hızı, dakikada), "egzersiz anjinası" (egzersiz testi anjinaya yol açtıysa 1), "ST çökmesi" (egzersizde ST segmentinin istirahate göre çökmesi, mm) ve "göğüs ağrısı tipi" ("tipik anjina", "atipik anjina", "anjina dışı ağrı" ya da "ağrı yok"). "hastalık" anjiyogramda koroner arter hastalığı görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Tabloyu bu sırayla bir modelin sütunlarına dönüştür: Yaş, erkek için 1 ve kadın için 0 olan cinsiyet, istirahat kan basıncı, kolesterol, en yüksek kalp hızı, egzersiz anjinası, ST çökmesi ve tipik anjinayı referans alan, atipik anjina, anjina dışı ağrı ve ağrı yok için 0 ve 1 değerli üç sütun. Satırları tablonun sırasında tut. Eğitim hastalarında 300 ağaçlı bir rastgele orman kur. Sonucun tekrarlanabilmesi için rastgelelik tohumunu 2026 olarak sabitle. Sekiz klinik değişkeni sekiz oyuncu olarak ele al; göğüs ağrısı tipinin üç sütunu her zaman tek bir oyuncu olarak birlikte hareket etsin. 172 numaralı hasta için ormanın verdiği olasılığın kesin Shapley değerlerini hesapla: 256 oyuncu grubunun her biri için gruptaki oyunculara hastanın değerlerini, öteki oyunculara sırayla her eğitim hastasının değerlerini ver, tahmin et ve eğitim hastaları üzerinden ortalama al; bu, grubun değeridir. Bir oyuncunun Shapley değeri, onu içermeyen bütün gruplar üzerinden, grubun oyuncuyla birlikteki değeri eksi oyuncu olmadan değerinin s!(7 − s)!/8! ağırlığıyla çarpımlarının toplamıdır; burada s, oyuncu olmadan grubun büyüklüğüdür. Sekiz değerin toplamının hastanın tahmini eksi boş grubun değerine eşit olduğunu kontrol et.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Sekiz Shapley değerini mutlak değeri en büyükten en küçüğe, boş grubun değeri ve tahminle birlikte göster; göğüs ağrısı tipinin Shapley değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Sekiz değerin toplamı, altıncı ondalık basamağa kadar tahmin eksi boş grubun değerine eşit olmalı.  

In [ ]:
#@title Adım 1.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _cad_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.2 kontrolü
teach.check_step("Adım 1.2", "cevap", globals())

In [ ]:
#@title Adım 1.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _cad_table()
d = veri
agri = d["göğüs ağrısı tipi"]
X = pd.DataFrame({"yaş": d["yaş"], "erkek": (d["cinsiyet"] == "erkek").astype(int),
                    "istirahat kan basıncı": d["istirahat kan basıncı"], "kolesterol": d["kolesterol"],
                    "en yüksek kalp hızı": d["en yüksek kalp hızı"], "egzersiz anjinası": d["egzersiz anjinası"],
                    "ST çökmesi": d["ST çökmesi"],
                    "atipik anjina": (agri == "atipik anjina").astype(int),
                    "anjina dışı ağrı": (agri == "anjina dışı ağrı").astype(int),
                    "ağrı yok": (agri == "ağrı yok").astype(int)})
y = d["hastalık"].to_numpy()
eg_ = (d["küme"] == "eğitim").to_numpy()
te_ = ~eg_
oyuncular = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

orman = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X[eg_], y[eg_])
lojistik = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[eg_], y[eg_])


def degerler(model, x, arka_plan):
    """Bir hasta için 256 grubun değerleri: Arka plan hastaları üzerinden ortalama olasılık."""
    maskeler = list(itertools.product([0, 1], repeat=8))
    satirlar = []
    for maske in maskeler:
        tut = np.zeros(10, dtype=bool)
        for m, sut in zip(maske, oyuncular):
            if m:
                tut[sut] = True
        satirlar.append(np.where(tut, x, arka_plan))
    p = model.predict_proba(pd.DataFrame(np.vstack(satirlar), columns=X.columns))[:, 1]
    return dict(zip(maskeler, p.reshape(len(maskeler), len(arka_plan)).mean(axis=1)))


def shapley(v):
    phi = np.zeros(8)
    for maske, deger in v.items():
        s = sum(maske)
        for j in range(8):
            if maske[j] == 0:
                j_ile = maske[:j] + (1,) + maske[j + 1:]
                phi[j] += math.factorial(s) * math.factorial(7 - s) / math.factorial(8) * (v[j_ile] - deger)
    return phi


arka_plan = X[eg_].to_numpy(dtype=float)
adlar = ["yaş", "cinsiyet", "istirahat kan basıncı", "kolesterol", "en yüksek kalp hızı",
               "egzersiz anjinası", "ST çökmesi", "göğüs ağrısı tipi"]

x = X[d["hasta"] == 172].to_numpy(dtype=float)[0]
v = degerler(orman, x, arka_plan)
phi = shapley(v)
print(pd.Series(phi, index=adlar).sort_values(key=abs, ascending=False).round(3))
print("boş grup", round(v[(0,) * 8], 3), "| tahmin", round(v[(1,) * 8], 3), "| toplam", round(phi.sum(), 6))
cevap = round(float(phi[7]), 3)
teach.check_step("Adım 1.2", "cevap", globals())

**Sonuç ne gösteriyor?** Göğüs ağrısı tipi 172 numaralı hastanın riskine 0,155 ekler: Bu veride istirahatte ağrısı olmadığı hâlde anjiyografiye gönderilen bir hastada çoğu zaman hastalık vardır. Kadın olmak ve ST çökmesinin olmaması ise riskini aşağı çeker.

### Adım 1.3: Lojistik regresyonun kapalı formu

**Amaç.** Bu adımda lojistik regresyonun 172 numaralı hasta için log-odds ölçeğindeki Shapley değerleri kapalı bir formla hesaplanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kurul, lojistik regresyonun 172 numaralı hastaya neden ormandan daha düşük bir olasılık verdiğini ve iki modelin hangi değişkenlerde ayrıştığını görmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani koroner anjiyografiye gönderilmiş hastalardan 303 hasta bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını (0-302), "küme" 2. haftanın kümesini verir ("eğitim", 227 hasta; "test", 76 hasta). Klinik değişkenler şunlardır: "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), "istirahat kan basıncı" (mmHg), "kolesterol" (mg/dL), "en yüksek kalp hızı" (egzersiz testinde ulaşılan en yüksek kalp hızı, dakikada), "egzersiz anjinası" (egzersiz testi anjinaya yol açtıysa 1), "ST çökmesi" (egzersizde ST segmentinin istirahate göre çökmesi, mm) ve "göğüs ağrısı tipi" ("tipik anjina", "atipik anjina", "anjina dışı ağrı" ya da "ağrı yok"). "hastalık" anjiyogramda koroner arter hastalığı görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Tabloyu bu sırayla bir modelin sütunlarına dönüştür: Yaş, erkek için 1 ve kadın için 0 olan cinsiyet, istirahat kan basıncı, kolesterol, en yüksek kalp hızı, egzersiz anjinası, ST çökmesi ve tipik anjinayı referans alan, atipik anjina, anjina dışı ağrı ve ağrı yok için 0 ve 1 değerli üç sütun. Satırları tablonun sırasında tut. Eğitim hastalarında, sütunları eğitim hastalarıyla standartlaştırılmış ve en fazla 1.000 yinelemeli bir lojistik regresyon kur. Log-odds ölçeğinde bir değişkenin bir hasta için Shapley değeri, özgün sütunun birimi başına katsayısı ile hastanın değeri eksi eğitim hastalarının ortalamasının çarpımıdır; göğüs ağrısı tipi için sütunlarının üç terimini topla. 172 numaralı hasta için sekiz değeri hesapla ve toplamlarının hastanın log-odds değeri eksi eğitim hastalarının ortalama log-odds değerine eşit olduğunu kontrol et.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Sekiz değeri değişkenleriyle birlikte göster; cinsiyetin değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Sekiz değerin toplamı, hastanın log-odds değeri eksi eğitim hastalarının ortalama log-odds değerine eşit olmalı.  

In [ ]:
#@title Adım 1.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _cad_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.3 kontrolü
teach.check_step("Adım 1.3", "cevap", globals())

In [ ]:
#@title Adım 1.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _cad_table()
d = veri
agri = d["göğüs ağrısı tipi"]
X = pd.DataFrame({"yaş": d["yaş"], "erkek": (d["cinsiyet"] == "erkek").astype(int),
                    "istirahat kan basıncı": d["istirahat kan basıncı"], "kolesterol": d["kolesterol"],
                    "en yüksek kalp hızı": d["en yüksek kalp hızı"], "egzersiz anjinası": d["egzersiz anjinası"],
                    "ST çökmesi": d["ST çökmesi"],
                    "atipik anjina": (agri == "atipik anjina").astype(int),
                    "anjina dışı ağrı": (agri == "anjina dışı ağrı").astype(int),
                    "ağrı yok": (agri == "ağrı yok").astype(int)})
y = d["hastalık"].to_numpy()
eg_ = (d["küme"] == "eğitim").to_numpy()
te_ = ~eg_
oyuncular = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

orman = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X[eg_], y[eg_])
lojistik = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[eg_], y[eg_])

olcekleyici, lr = lojistik[0], lojistik[1]
beta = lr.coef_[0] / olcekleyici.scale_
x = X[d["hasta"] == 172].to_numpy(dtype=float)[0]
terimler = beta * (x - X[eg_].mean().to_numpy())
phi = np.array([terimler[sut].sum() for sut in oyuncular])
adlar = ["yaş", "cinsiyet", "istirahat kan basıncı", "kolesterol", "en yüksek kalp hızı",
               "egzersiz anjinası", "ST çökmesi", "göğüs ağrısı tipi"]
print(pd.Series(phi, index=adlar).round(3))
z = lojistik.decision_function(X)
print("kontrol:", round(phi.sum(), 6), round(z[d["hasta"].to_numpy() == 172][0] - z[eg_].mean(), 6))
cevap = round(float(phi[1]), 3)
teach.check_step("Adım 1.3", "cevap", globals())

**Sonuç ne gösteriyor?** Kadın olmak 172 numaralı hastanın log-odds değerine −1,175 ekler ve bu, sekiz değerin mutlak değerce en büyüğüdür. Ormanda ise cinsiyet yalnızca üçüncü sıradadır. Lojistik regresyon cinsiyete her hasta için büyük bir katsayı verir; orman ise cinsiyeti yalnızca ağaçlarının bir kısmında kullanır.

### Adım 1.4: Tek bir hastadan bütün hastalara

**Amaç.** Bu adımda ormanın her test hastası için Shapley değerleri hesaplanır ve değişkenler sıralanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kurul, ormanın bütün test hastalarında hangi klinik değişkenlere dayandığını ve bunlardan herhangi birinin hastalıktan çok hastaların nasıl sevk edildiğini yansıtıp yansıtmadığını öğrenmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani koroner anjiyografiye gönderilmiş hastalardan 303 hasta bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını (0-302), "küme" 2. haftanın kümesini verir ("eğitim", 227 hasta; "test", 76 hasta). Klinik değişkenler şunlardır: "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), "istirahat kan basıncı" (mmHg), "kolesterol" (mg/dL), "en yüksek kalp hızı" (egzersiz testinde ulaşılan en yüksek kalp hızı, dakikada), "egzersiz anjinası" (egzersiz testi anjinaya yol açtıysa 1), "ST çökmesi" (egzersizde ST segmentinin istirahate göre çökmesi, mm) ve "göğüs ağrısı tipi" ("tipik anjina", "atipik anjina", "anjina dışı ağrı" ya da "ağrı yok"). "hastalık" anjiyogramda koroner arter hastalığı görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Tabloyu bu sırayla bir modelin sütunlarına dönüştür: Yaş, erkek için 1 ve kadın için 0 olan cinsiyet, istirahat kan basıncı, kolesterol, en yüksek kalp hızı, egzersiz anjinası, ST çökmesi ve tipik anjinayı referans alan, atipik anjina, anjina dışı ağrı ve ağrı yok için 0 ve 1 değerli üç sütun. Satırları tablonun sırasında tut. Eğitim hastalarında 300 ağaçlı bir rastgele orman kur. Sonucun tekrarlanabilmesi için rastgelelik tohumunu 2026 olarak sabitle. Sekiz klinik değişkeni sekiz oyuncu olarak ele al; göğüs ağrısı tipinin üç sütunu her zaman tek bir oyuncu olarak birlikte hareket etsin. Her test hastası için ormanın verdiği olasılığın kesin Shapley değerlerini hesapla: 256 oyuncu grubunun her biri için gruptaki oyunculara hastanın değerlerini, öteki oyunculara sırayla her eğitim hastasının değerlerini ver, tahmin et ve eğitim hastaları üzerinden ortalama al; bu, grubun değeridir. Bir oyuncunun Shapley değeri, onu içermeyen bütün gruplar üzerinden, grubun oyuncuyla birlikteki değeri eksi oyuncu olmadan değerinin s!(7 − s)!/8! ağırlığıyla çarpımlarının toplamıdır; burada s, oyuncu olmadan grubun büyüklüğüdür. Sekiz değerin toplamının hastanın tahmini eksi boş grubun değerine eşit olduğunu kontrol et. Ardından her değişken için test hastaları üzerinden mutlak Shapley değerlerinin ortalamasını hesapla ve her değişkenin en büyük mutlak değere sahip olduğu test hastalarını say.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Ortalama mutlak değerleri en büyükten en küçüğe, sayılarla birlikte göster; göğüs ağrısı tipinin ortalama mutlak değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Sayıların toplamı test hastalarının sayısına eşit olmalı.  

In [ ]:
#@title Adım 1.4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _cad_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.4 kontrolü
teach.check_step("Adım 1.4", "cevap", globals())

In [ ]:
#@title Adım 1.4 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _cad_table()
d = veri
agri = d["göğüs ağrısı tipi"]
X = pd.DataFrame({"yaş": d["yaş"], "erkek": (d["cinsiyet"] == "erkek").astype(int),
                    "istirahat kan basıncı": d["istirahat kan basıncı"], "kolesterol": d["kolesterol"],
                    "en yüksek kalp hızı": d["en yüksek kalp hızı"], "egzersiz anjinası": d["egzersiz anjinası"],
                    "ST çökmesi": d["ST çökmesi"],
                    "atipik anjina": (agri == "atipik anjina").astype(int),
                    "anjina dışı ağrı": (agri == "anjina dışı ağrı").astype(int),
                    "ağrı yok": (agri == "ağrı yok").astype(int)})
y = d["hastalık"].to_numpy()
eg_ = (d["küme"] == "eğitim").to_numpy()
te_ = ~eg_
oyuncular = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

orman = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X[eg_], y[eg_])
lojistik = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[eg_], y[eg_])


def degerler(model, x, arka_plan):
    """Bir hasta için 256 grubun değerleri: Arka plan hastaları üzerinden ortalama olasılık."""
    maskeler = list(itertools.product([0, 1], repeat=8))
    satirlar = []
    for maske in maskeler:
        tut = np.zeros(10, dtype=bool)
        for m, sut in zip(maske, oyuncular):
            if m:
                tut[sut] = True
        satirlar.append(np.where(tut, x, arka_plan))
    p = model.predict_proba(pd.DataFrame(np.vstack(satirlar), columns=X.columns))[:, 1]
    return dict(zip(maskeler, p.reshape(len(maskeler), len(arka_plan)).mean(axis=1)))


def shapley(v):
    phi = np.zeros(8)
    for maske, deger in v.items():
        s = sum(maske)
        for j in range(8):
            if maske[j] == 0:
                j_ile = maske[:j] + (1,) + maske[j + 1:]
                phi[j] += math.factorial(s) * math.factorial(7 - s) / math.factorial(8) * (v[j_ile] - deger)
    return phi


arka_plan = X[eg_].to_numpy(dtype=float)
adlar = ["yaş", "cinsiyet", "istirahat kan basıncı", "kolesterol", "en yüksek kalp hızı",
               "egzersiz anjinası", "ST çökmesi", "göğüs ağrısı tipi"]

S = np.array([shapley(degerler(orman, xi, arka_plan)) for xi in X[te_].to_numpy(dtype=float)])
tablo = pd.DataFrame({"ortalama mutlak değer": np.abs(S).mean(axis=0),
                            "en büyük olduğu hasta": np.bincount(np.abs(S).argmax(axis=1), minlength=8)}, index=adlar)
print(tablo.sort_values("ortalama mutlak değer", ascending=False).round(3))
cevap = round(float(np.abs(S).mean(axis=0)[7]), 3)
teach.check_step("Adım 1.4", "cevap", globals())

**Sonuç ne gösteriyor?** Göğüs ağrısı tipi 0,138 ortalama mutlak değerle başı çeker. Model, ağrısı olmadığı hâlde anjiyografiye gönderilen hastalarda sıklıkla hastalık bulunduğunu öğrenmiştir; bir kurul bunu bir neden olarak değil, verinin bir özelliği olarak okur.

**1. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#stop-1) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 2: Tek bir hasta için LIME ve eko ağının baktığı yer

*Ders: [2. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#stop-2)* · *Python yolunda aynı sonuç: Kısım 2, 4-7. bölümler*

Bu durak tahminleri iki yolla daha açıklar. LIME, 1. duraktaki ormanı 172 numaralı hastanın çevresinde ağırlıklı doğrusal bir modelle taklit eder; yirmi tohumla yapılan bir tekrar da cevabının ne kadar kararlı olduğunu gösterir. Ardından küçük bir evrişimli ağ, CAMUS ekokardiyogramlarında yüzde 45'in altındaki ejeksiyon fraksiyonunu işaretlemeyi öğrenir ve ağın Grad-CAM haritaları sol ventrikülün uzman sınırlarıyla karşılaştırılır.

**Durağın sonunda:** 172 numaralı hasta için göğüs ağrısı tipinin LIME katsayısını (0,145), yalnızca 100 komşuyla göğüs ağrısı tipini ilk sıraya koyan tohumların yirmi tohum içindeki payını (0,70), eko ağının test AUC değerini (0,796) ve Grad-CAM haritalarının uzman sınırı içindeki ortalama payını (0,317) bulacaksınız.

In [ ]:
#@title 2. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
_heart = load_cleveland()
# 2. haftanın ayırması, kendi sırasıyla: Önce eğitim, sonra test hastaları
_cad_tr, _cad_te = train_test_split(_heart.index.to_numpy(), test_size=0.25, stratify=_heart["disease"], random_state=RANDOM_STATE)
_CP = {1.0: "tipik anjina", 2.0: "atipik anjina", 3.0: "anjina dışı ağrı", 4.0: "ağrı yok"}


def _cad_table(order="split"):
    """Cleveland verisindeki 303 hasta; kümeleri, sekiz klinik değişken ve hastalıkla birlikte; 2. haftanın ayırmasının ya da hasta numaralarının sırasıyla."""
    h = _heart.loc[list(_cad_tr) + list(_cad_te)] if order == "split" else _heart
    return pd.DataFrame({"hasta": h.index, "küme": np.where(h.index.isin(_cad_tr), "eğitim", "test"),
                         "yaş": h["age"].astype(int), "cinsiyet": h["sex"].map({1.0: "erkek", 0.0: "kadın"}),
                         "istirahat kan basıncı": h["trestbps"].astype(int), "kolesterol": h["chol"].astype(int),
                         "en yüksek kalp hızı": h["thalach"].astype(int), "egzersiz anjinası": h["exang"].astype(int),
                         "ST çökmesi": h["oldpeak"], "göğüs ağrısı tipi": h["cp"].map(_CP),
                         "hastalık": h["disease"]}).reset_index(drop=True)

_SETS_E = {"training": "eğitim", "validation": "doğrulama", "testing": "test"}


def _echo():
    """Dersin CAMUS görüntüleri; kareleri ve uzman sınırlarıyla birlikte (bir kez yüklenir, birkaç saniye sürer)."""
    if "_echo_cache" not in globals():
        frames_ = camus.frames(camus.patients(), size=96)
        info = camus.table()
        low_ef = dict(zip(info["patient"], (info["ef"] < 45).astype(int)))
        index = {key: i for i, key in enumerate(zip(frames_["patient"], frames_["view"], frames_["phase"]))}
        views = sorted(set(zip(frames_["patient"], frames_["view"])))     # 300 hasta x 2 görüntü
        echo_x = np.stack([[frames_["images"][index[(p, v, "ED")]], frames_["images"][index[(p, v, "ES")]]] for p, v in views]).astype(np.float32)
        labels = [np.isin(frames_["masks"][index[(p, v, phase)]], (1, 2)) for p, v in views for phase in ("ED", "ES")]
        outline = np.array(labels).reshape(len(views), 2, 96, 96).any(axis=1)   # ED ya da ES karesinde kavite ya da miyokard
        echo_y = np.array([low_ef[p] for p, v in views])
        echo_set = np.array([frames_["split"][index[(p, v, "ED")]] for p, v in views])
        etr, eva, ete = echo_set == "training", echo_set == "validation", echo_set == "testing"
        globals()["_echo_cache"] = dict(views=views, echo_x=echo_x, outline=outline, echo_y=echo_y, echo_set=echo_set,
                                        quality=camus.table().set_index("patient"))
    return _echo_cache


def _echo_table():
    """Dersin 300 CAMUS hastasının 600 görüntüsü; kareleri, uzman sınırı ve yüzde 45'in altındaki ejeksiyon fraksiyonuyla birlikte."""
    e = _echo()
    q = e["quality"]
    return pd.DataFrame({"hasta": [p for p, v in e["views"]], "görüntü": [v for p, v in e["views"]],
                         "küme": [_SETS_E[s] for s in e["echo_set"]],
                         "görüntü kalitesi": [q.loc[p, "quality_2ch" if v == "2CH" else "quality_4ch"] for p, v in e["views"]],
                         "diyastol sonu": list(e["echo_x"][:, 0]), "sistol sonu": list(e["echo_x"][:, 1]),
                         "sınır": list(e["outline"].astype(np.uint8)), "düşük ejeksiyon fraksiyonu": e["echo_y"]})


print(len(_heart), "hasta Cleveland verisinde,", len(_echo()["views"]), "görüntü CAMUS'ta")
print("2. durak hazır.")

### Adım 2.1: 172 numaralı hasta için LIME

**Amaç.** Bu adımda ormanın 172 numaralı hastaya verdiği olasılık, hastanın çevresine uydurulan ağırlıklı doğrusal bir modelle açıklanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kurul, 172 numaralı hastanın tahmini için Shapley değerlerinden farklı bir yolla yapılmış ikinci bir açıklama görmek ve ikisinin uyuşup uyuşmadığını anlamak istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani koroner anjiyografiye gönderilmiş hastalardan 303 hasta bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını (0-302), "küme" 2. haftanın kümesini verir ("eğitim", 227 hasta; "test", 76 hasta). Klinik değişkenler şunlardır: "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), "istirahat kan basıncı" (mmHg), "kolesterol" (mg/dL), "en yüksek kalp hızı" (egzersiz testinde ulaşılan en yüksek kalp hızı, dakikada), "egzersiz anjinası" (egzersiz testi anjinaya yol açtıysa 1), "ST çökmesi" (egzersizde ST segmentinin istirahate göre çökmesi, mm) ve "göğüs ağrısı tipi" ("tipik anjina", "atipik anjina", "anjina dışı ağrı" ya da "ağrı yok"). "hastalık" anjiyogramda koroner arter hastalığı görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Tabloyu bu sırayla bir modelin sütunlarına dönüştür: Yaş, erkek için 1 ve kadın için 0 olan cinsiyet, istirahat kan basıncı, kolesterol, en yüksek kalp hızı, egzersiz anjinası, ST çökmesi ve tipik anjinayı referans alan, atipik anjina, anjina dışı ağrı ve ağrı yok için 0 ve 1 değerli üç sütun. Satırları tablonun sırasında tut. Eğitim hastalarında 300 ağaçlı bir rastgele orman kur. Sonucun tekrarlanabilmesi için rastgelelik tohumunu 2026 olarak sabitle. Sekiz klinik değişkeni sekiz oyuncu olarak ele al; göğüs ağrısı tipinin üç sütunu her zaman tek bir oyuncu olarak birlikte hareket etsin. Ormanın 172 numaralı hastaya verdiği olasılığı LIME ile açıkla. Hastanın komşularını çek: Her komşu için rastgele bir eğitim hastası çek; sekiz oyuncunun her biri yarı olasılıkla 172 numaralı hastanın değerini korusun, aksi hâlde çekilen eğitim hastasının değerini alsın. Her komşu için hangi oyuncuların hastanın değerini koruduğunu (1) ve hangilerinin değiştirildiğini (0) kaydet. Her komşuyu exp(−d²/w²) ile ağırlıklandır; burada d değiştirilen oyuncu sayısının karekökü, w = 0,75 × √8 olsun. Komşular için ormanın verdiği olasılıkları sekiz adet 0-1 sütunu üzerine bu ağırlıklarla, cezası 1 olan bir ridge regresyonuyla modelle; sekiz katsayı açıklamadır. Tek bir rastgele sayı üreteciyle önce bütün komşuların 0 ve 1 değerlerini, ardından eğitim hastalarını çek. 2.000 komşu ve 2026 tohumunu kullan.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Sekiz katsayıyı mutlak değeri en büyükten en küçüğe göster; göğüs ağrısı tipinin katsayısını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Hastanın sekiz değerinin hepsini koruyan bir komşunun ağırlığı 1 olmalı.  

In [ ]:
#@title Adım 2.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _cad_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.1 kontrolü
teach.check_step("Adım 2.1", "cevap", globals())

In [ ]:
#@title Adım 2.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _cad_table()
d = veri
agri = d["göğüs ağrısı tipi"]
X = pd.DataFrame({"yaş": d["yaş"], "erkek": (d["cinsiyet"] == "erkek").astype(int),
                    "istirahat kan basıncı": d["istirahat kan basıncı"], "kolesterol": d["kolesterol"],
                    "en yüksek kalp hızı": d["en yüksek kalp hızı"], "egzersiz anjinası": d["egzersiz anjinası"],
                    "ST çökmesi": d["ST çökmesi"],
                    "atipik anjina": (agri == "atipik anjina").astype(int),
                    "anjina dışı ağrı": (agri == "anjina dışı ağrı").astype(int),
                    "ağrı yok": (agri == "ağrı yok").astype(int)})
y = d["hastalık"].to_numpy()
eg_ = (d["küme"] == "eğitim").to_numpy()
te_ = ~eg_
oyuncular = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

orman = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X[eg_], y[eg_])
lojistik = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[eg_], y[eg_])

arka_plan = X[eg_].to_numpy(dtype=float)

def lime(n, tohum):
    rng = np.random.default_rng(tohum)
    Z = rng.integers(0, 2, size=(n, 8))
    satirlar = arka_plan[rng.integers(0, len(arka_plan), size=n)].copy()
    for j, sut in enumerate(oyuncular):
        satirlar[:, sut] = np.where(Z[:, [j]] == 1, x[sut], satirlar[:, sut])
    p = orman.predict_proba(pd.DataFrame(satirlar, columns=X.columns))[:, 1]
    d2 = (1 - Z).sum(axis=1)
    w = np.exp(-d2 / (0.75 * np.sqrt(8)) ** 2)
    return Ridge(alpha=1.0).fit(Z, p, sample_weight=w).coef_


x = X[d["hasta"] == 172].to_numpy(dtype=float)[0]

adlar = ["yaş", "cinsiyet", "istirahat kan basıncı", "kolesterol", "en yüksek kalp hızı",
               "egzersiz anjinası", "ST çökmesi", "göğüs ağrısı tipi"]
katsayi = lime(2000, RANDOM_STATE)
print(pd.Series(katsayi, index=adlar).sort_values(key=abs, ascending=False).round(3))
cevap = round(float(katsayi[7]), 3)
teach.check_step("Adım 2.1", "cevap", globals())

**Sonuç ne gösteriyor?** LIME göğüs ağrısı tipine 0,145 katsayı verir; bu değer 1. duraktaki Shapley değerine yakındır. Hastanın değerlerini koruyan ya da değiştiren komşularla LIME ve Shapley değerleri neredeyse aynı soruyu cevaplar.

### Adım 2.2: LIME ne kadar kararlı?

**Amaç.** Bu adımda LIME az sayıda komşu ve yirmi tohumla tekrarlanır ve başı çeken değişkenin ne sıklıkla değiştiği görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kurul, tek bir hastanın açıklamasına güvenmeden önce LIME ile yapılan bir açıklamanın yeniden çalıştırıldığında aynı kalıp kalmayacağını öğrenmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani koroner anjiyografiye gönderilmiş hastalardan 303 hasta bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını (0-302), "küme" 2. haftanın kümesini verir ("eğitim", 227 hasta; "test", 76 hasta). Klinik değişkenler şunlardır: "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), "istirahat kan basıncı" (mmHg), "kolesterol" (mg/dL), "en yüksek kalp hızı" (egzersiz testinde ulaşılan en yüksek kalp hızı, dakikada), "egzersiz anjinası" (egzersiz testi anjinaya yol açtıysa 1), "ST çökmesi" (egzersizde ST segmentinin istirahate göre çökmesi, mm) ve "göğüs ağrısı tipi" ("tipik anjina", "atipik anjina", "anjina dışı ağrı" ya da "ağrı yok"). "hastalık" anjiyogramda koroner arter hastalığı görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Tabloyu bu sırayla bir modelin sütunlarına dönüştür: Yaş, erkek için 1 ve kadın için 0 olan cinsiyet, istirahat kan basıncı, kolesterol, en yüksek kalp hızı, egzersiz anjinası, ST çökmesi ve tipik anjinayı referans alan, atipik anjina, anjina dışı ağrı ve ağrı yok için 0 ve 1 değerli üç sütun. Satırları tablonun sırasında tut. Eğitim hastalarında 300 ağaçlı bir rastgele orman kur. Sonucun tekrarlanabilmesi için rastgelelik tohumunu 2026 olarak sabitle. Sekiz klinik değişkeni sekiz oyuncu olarak ele al; göğüs ağrısı tipinin üç sütunu her zaman tek bir oyuncu olarak birlikte hareket etsin. Ormanın 172 numaralı hastaya verdiği olasılığı LIME ile açıkla. Hastanın komşularını çek: Her komşu için rastgele bir eğitim hastası çek; sekiz oyuncunun her biri yarı olasılıkla 172 numaralı hastanın değerini korusun, aksi hâlde çekilen eğitim hastasının değerini alsın. Her komşu için hangi oyuncuların hastanın değerini koruduğunu (1) ve hangilerinin değiştirildiğini (0) kaydet. Her komşuyu exp(−d²/w²) ile ağırlıklandır; burada d değiştirilen oyuncu sayısının karekökü, w = 0,75 × √8 olsun. Komşular için ormanın verdiği olasılıkları sekiz adet 0-1 sütunu üzerine bu ağırlıklarla, cezası 1 olan bir ridge regresyonuyla modelle; sekiz katsayı açıklamadır. Tek bir rastgele sayı üreteciyle önce bütün komşuların 0 ve 1 değerlerini, ardından eğitim hastalarını çek. Yalnızca 100 komşu kullan ve 1'den 20'ye kadar tohumlarla tekrarla. Her tohum için mutlak katsayısı en büyük olan oyuncuyu bul.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her oyuncunun kaç kez ilk sıraya geldiğini ve göğüs ağrısı tipinin en küçük ve en büyük katsayısını göster; göğüs ağrısı tipinin ilk sıraya geldiği tohumların payını iki ondalık basamakla "cevap" adıyla sakla.  
> Test: Oyuncuların sayılarının toplamı 20 olmalı.  

In [ ]:
#@title Adım 2.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _cad_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.2 kontrolü
teach.check_step("Adım 2.2", "cevap", globals())

In [ ]:
#@title Adım 2.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _cad_table()
d = veri
agri = d["göğüs ağrısı tipi"]
X = pd.DataFrame({"yaş": d["yaş"], "erkek": (d["cinsiyet"] == "erkek").astype(int),
                    "istirahat kan basıncı": d["istirahat kan basıncı"], "kolesterol": d["kolesterol"],
                    "en yüksek kalp hızı": d["en yüksek kalp hızı"], "egzersiz anjinası": d["egzersiz anjinası"],
                    "ST çökmesi": d["ST çökmesi"],
                    "atipik anjina": (agri == "atipik anjina").astype(int),
                    "anjina dışı ağrı": (agri == "anjina dışı ağrı").astype(int),
                    "ağrı yok": (agri == "ağrı yok").astype(int)})
y = d["hastalık"].to_numpy()
eg_ = (d["küme"] == "eğitim").to_numpy()
te_ = ~eg_
oyuncular = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

orman = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X[eg_], y[eg_])
lojistik = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X[eg_], y[eg_])

arka_plan = X[eg_].to_numpy(dtype=float)

def lime(n, tohum):
    rng = np.random.default_rng(tohum)
    Z = rng.integers(0, 2, size=(n, 8))
    satirlar = arka_plan[rng.integers(0, len(arka_plan), size=n)].copy()
    for j, sut in enumerate(oyuncular):
        satirlar[:, sut] = np.where(Z[:, [j]] == 1, x[sut], satirlar[:, sut])
    p = orman.predict_proba(pd.DataFrame(satirlar, columns=X.columns))[:, 1]
    d2 = (1 - Z).sum(axis=1)
    w = np.exp(-d2 / (0.75 * np.sqrt(8)) ** 2)
    return Ridge(alpha=1.0).fit(Z, p, sample_weight=w).coef_


x = X[d["hasta"] == 172].to_numpy(dtype=float)[0]

adlar = ["yaş", "cinsiyet", "istirahat kan basıncı", "kolesterol", "en yüksek kalp hızı",
               "egzersiz anjinası", "ST çökmesi", "göğüs ağrısı tipi"]
calistirmalar = np.array([lime(100, s) for s in range(1, 21)])
ilk = [adlar[j] for j in np.abs(calistirmalar).argmax(axis=1)]
print(pd.Series(ilk).value_counts())
print("göğüs ağrısı tipi", round(calistirmalar[:, 7].min(), 3), "ile", round(calistirmalar[:, 7].max(), 3))
cevap = round(float(np.mean([f == adlar[7] for f in ilk])), 2)
teach.check_step("Adım 2.2", "cevap", globals())

**Sonuç ne gösteriyor?** 100 komşuyla göğüs ağrısı tipi tohumların 0,70 kadarında ilk sırada yer alır; öteki tohumlar egzersiz anjinasını ya da başka bir değişkeni ilk sıraya koyar. LIME'a dayanan bir rapor komşu sayısını belirtir ve sonucun tohumlar arasında korunduğunu gösterir.

### Adım 2.3: Düşük ejeksiyon fraksiyonu için bir ağ

**Amaç.** Bu adımda tek bir görüntüde yüzde 45'in altındaki ejeksiyon fraksiyonunu işaretleyen küçük bir evrişimli ağ eğitilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kurul ekokardiyogramları okuyan bir ağı inceliyor. Ağın haritaları değerlendirilmeden önce, görmediği hastaların görüntülerinde düşük ejeksiyon fraksiyonunu ne kadar iyi işaretlediğini öğrenmek istiyorum.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda herkese açık bir ekokardiyogram veri seti olan CAMUS'tan 600 görüntü, yani 300 hastanın apikal iki boşluk ve dört boşluk görüntüleri bulunur. Her satır bir görüntüdür: "hasta" hastayı, "görüntü" görüntüyü ("2CH" ya da "4CH"), "küme" hastanın kümesini ("eğitim", 200 hasta; "doğrulama", 50 hasta; "test", 50 hasta), "görüntü kalitesi" ise uzmanların verdiği derecesini ("Good", "Medium" ya da "Poor") verir. "diyastol sonu" ve "sistol sonu" sütunlarının her hücresi kareyi 0 ile 1 arasında değerler alan 96 × 96 piksellik bir görüntü olarak tutar. "sınır" sütununun her hücresinde ise sol ventrikül kavitesinin ve miyokardın diyastol sonu ya da sistol sonundaki uzman sınırı aynı boyutta yer alır: İçeride 1, dışarıda 0. "düşük ejeksiyon fraksiyonu" hastanın ejeksiyon fraksiyonu yüzde 45'in altındaysa 1, aksi hâlde 0 olur.  
> Görev: Tek bir görüntü için diyastol sonu ve sistol sonu karelerini 96 × 96 piksellik iki kanal olarak alan küçük bir evrişimli ağ kur. Ağda dört blok bulunsun; her blok, boyutu koruyan 3 × 3 çekirdekli bir evrişimden (sırasıyla 16, 32, 64 ve 64 kanal), yığın normalleştirmesinden, ReLU'dan ve 2 × 2 en büyük değer havuzlamasından oluşsun. Bloklardan sonra ağ her kanalın görüntü üzerindeki ortalamasını alsın ve 0,3 oranında seyreltme uygulasın; tek çıktılı bir doğrusal katman da düşük ejeksiyon fraksiyonunun logitini versin. Ağı eğitim görüntülerinde Adam (öğrenme hızı 0,001, ağırlık azaltma 0,0001) ve ikili çapraz entropiyle, 32 görüntülük karıştırılmış gruplarla 30 dönem eğit. Her grubu her yönde en fazla 4 piksel rastgele kaydır ve doğrulama görüntülerinde AUC değeri en iyi olan dönemin ağırlıklarını sakla. Ağ kütüphanesinin rastgele sayı üretecini ağı kurmadan önce ve eğitimden önce yeniden 2026 ile başlat. Test görüntülerindeki AUC değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: En iyi doğrulama AUC değerini ve test AUC değerini yazdır; test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Dönemi seçmek için hiçbir test görüntüsü kullanılmamalı ve bir hastanın görüntüleri tek bir kümede kalmalı.  

In [ ]:
#@title Adım 2.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _echo_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.3 kontrolü
teach.check_step("Adım 2.3", "cevap", globals())

In [ ]:
#@title Adım 2.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _echo_table()
d = veri
Xe = np.stack([np.stack(d["diyastol sonu"]), np.stack(d["sistol sonu"])], axis=1).astype(np.float32)
ye = d["düşük ejeksiyon fraksiyonu"].to_numpy()
eeg_, edo_, ete_ = [(d["küme"] == k).to_numpy() for k in ("eğitim", "doğrulama", "test")]


class EkoAgi(nn.Module):
    def __init__(self):
        super().__init__()
        c = [2, 16, 32, 64, 64]
        self.blocks = nn.Sequential(*[nn.Sequential(nn.Conv2d(c[i], c[i + 1], 3, padding=1), nn.BatchNorm2d(c[i + 1]), nn.ReLU(), nn.MaxPool2d(2))
                                      for i in range(4)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(x)).squeeze(1)


@torch.no_grad()
def tahmin(ag, X):
    ag.eval()
    return torch.cat([torch.sigmoid(ag(torch.as_tensor(X[i:i + 64]))) for i in range(0, len(X), 64)]).numpy()


def egit():
    torch.manual_seed(RANDOM_STATE)
    ag = EkoAgi()
    torch.manual_seed(RANDOM_STATE)
    Xt, yt = torch.as_tensor(Xe[eeg_]), torch.as_tensor(ye[eeg_], dtype=torch.float32)
    opt = torch.optim.Adam(ag.parameters(), lr=0.001, weight_decay=1e-4)
    kayip = nn.BCEWithLogitsLoss()
    en_iyi, durum = -1.0, None
    for donem in range(30):
        ag.train()
        sira = torch.randperm(len(Xt))
        for i in range(0, len(Xt), 32):
            b = sira[i:i + 32]
            dy, dx = (int(s) for s in torch.randint(-4, 5, (2,)))
            l = kayip(ag(torch.roll(Xt[b], shifts=(dy, dx), dims=(2, 3))), yt[b])
            opt.zero_grad()
            l.backward()
            opt.step()
        a = roc_auc_score(ye[edo_], tahmin(ag, Xe[edo_]))
        if a > en_iyi:
            en_iyi, durum = a, copy.deepcopy(ag.state_dict())
    ag.load_state_dict(durum)
    return ag

ag = egit()
p_test = tahmin(ag, Xe[ete_])
print("test AUC", round(roc_auc_score(ye[ete_], p_test), 3))
cevap = round(float(roc_auc_score(ye[ete_], p_test)), 3)
teach.check_step("Adım 2.3", "cevap", globals())

**Sonuç ne gösteriyor?** Ağ tek tek görüntülerde 0,796 test AUC değerine ulaşır: Düşük ejeksiyon fraksiyonlarının çoğunu işaretler, ancak 4. haftanın disk yönteminden çok uzaktır. Haritaları bir sonraki adımda değerlendirilir.

### Adım 2.4: Uzman sınırına karşı Grad-CAM haritaları

**Amaç.** Bu adımda her test görüntüsünün Grad-CAM haritası ve haritanın uzman sınırı içindeki payı hesaplanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kurul, ağın düşük ejeksiyon fraksiyonunu uzmanın baktığı sol ventrikülden mi, yoksa görüntünün başka yerlerinden mi işaretlediğini öğrenmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda herkese açık bir ekokardiyogram veri seti olan CAMUS'tan 600 görüntü, yani 300 hastanın apikal iki boşluk ve dört boşluk görüntüleri bulunur. Her satır bir görüntüdür: "hasta" hastayı, "görüntü" görüntüyü ("2CH" ya da "4CH"), "küme" hastanın kümesini ("eğitim", 200 hasta; "doğrulama", 50 hasta; "test", 50 hasta), "görüntü kalitesi" ise uzmanların verdiği derecesini ("Good", "Medium" ya da "Poor") verir. "diyastol sonu" ve "sistol sonu" sütunlarının her hücresi kareyi 0 ile 1 arasında değerler alan 96 × 96 piksellik bir görüntü olarak tutar. "sınır" sütununun her hücresinde ise sol ventrikül kavitesinin ve miyokardın diyastol sonu ya da sistol sonundaki uzman sınırı aynı boyutta yer alır: İçeride 1, dışarıda 0. "düşük ejeksiyon fraksiyonu" hastanın ejeksiyon fraksiyonu yüzde 45'in altındaysa 1, aksi hâlde 0 olur.  
> Görev: Tek bir görüntü için diyastol sonu ve sistol sonu karelerini 96 × 96 piksellik iki kanal olarak alan küçük bir evrişimli ağ kur. Ağda dört blok bulunsun; her blok, boyutu koruyan 3 × 3 çekirdekli bir evrişimden (sırasıyla 16, 32, 64 ve 64 kanal), yığın normalleştirmesinden, ReLU'dan ve 2 × 2 en büyük değer havuzlamasından oluşsun. Bloklardan sonra ağ her kanalın görüntü üzerindeki ortalamasını alsın ve 0,3 oranında seyreltme uygulasın; tek çıktılı bir doğrusal katman da düşük ejeksiyon fraksiyonunun logitini versin. Ağı eğitim görüntülerinde Adam (öğrenme hızı 0,001, ağırlık azaltma 0,0001) ve ikili çapraz entropiyle, 32 görüntülük karıştırılmış gruplarla 30 dönem eğit. Her grubu her yönde en fazla 4 piksel rastgele kaydır ve doğrulama görüntülerinde AUC değeri en iyi olan dönemin ağırlıklarını sakla. Ağ kütüphanesinin rastgele sayı üretecini ağı kurmadan önce ve eğitimden önce yeniden 2026 ile başlat. Her test görüntüsü için çıktının Grad-CAM haritasını son bloğun ReLU katmanında, havuzlamadan önce hesapla. Bunun için çıktının bu katmandaki 12 × 12 boyutlu 64 öznitelik haritasına göre gradyanını al, her kanalın ortalama gradyanını o kanalın ağırlığı olarak kullan ve 64 haritanın ağırlıklı toplamını oluştur. Toplamın yalnızca pozitif kısmını tut ve çift doğrusal aradeğerlemeyle 96 × 96 boyutuna büyüt. Her görüntü için haritanın toplamının sınırın içine düşen payını hesapla. Test görüntüleri üzerinden payların ortalamasını ve karşılaştırma için sınırın kapladığı görüntü payının ortalamasını bildir.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki ortalamayı yazdır; haritaların sınır içindeki ortalama payını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her pay 0 ile 1 arasında olmalı ve hiçbir harita eğitim kipindeki bir ağdan hesaplanmamalı.  

In [ ]:
#@title Adım 2.4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _echo_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.4 kontrolü
teach.check_step("Adım 2.4", "cevap", globals())

In [ ]:
#@title Adım 2.4 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _echo_table()
d = veri
Xe = np.stack([np.stack(d["diyastol sonu"]), np.stack(d["sistol sonu"])], axis=1).astype(np.float32)
ye = d["düşük ejeksiyon fraksiyonu"].to_numpy()
eeg_, edo_, ete_ = [(d["küme"] == k).to_numpy() for k in ("eğitim", "doğrulama", "test")]


class EkoAgi(nn.Module):
    def __init__(self):
        super().__init__()
        c = [2, 16, 32, 64, 64]
        self.blocks = nn.Sequential(*[nn.Sequential(nn.Conv2d(c[i], c[i + 1], 3, padding=1), nn.BatchNorm2d(c[i + 1]), nn.ReLU(), nn.MaxPool2d(2))
                                      for i in range(4)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(x)).squeeze(1)


@torch.no_grad()
def tahmin(ag, X):
    ag.eval()
    return torch.cat([torch.sigmoid(ag(torch.as_tensor(X[i:i + 64]))) for i in range(0, len(X), 64)]).numpy()


def egit():
    torch.manual_seed(RANDOM_STATE)
    ag = EkoAgi()
    torch.manual_seed(RANDOM_STATE)
    Xt, yt = torch.as_tensor(Xe[eeg_]), torch.as_tensor(ye[eeg_], dtype=torch.float32)
    opt = torch.optim.Adam(ag.parameters(), lr=0.001, weight_decay=1e-4)
    kayip = nn.BCEWithLogitsLoss()
    en_iyi, durum = -1.0, None
    for donem in range(30):
        ag.train()
        sira = torch.randperm(len(Xt))
        for i in range(0, len(Xt), 32):
            b = sira[i:i + 32]
            dy, dx = (int(s) for s in torch.randint(-4, 5, (2,)))
            l = kayip(ag(torch.roll(Xt[b], shifts=(dy, dx), dims=(2, 3))), yt[b])
            opt.zero_grad()
            l.backward()
            opt.step()
        a = roc_auc_score(ye[edo_], tahmin(ag, Xe[edo_]))
        if a > en_iyi:
            en_iyi, durum = a, copy.deepcopy(ag.state_dict())
    ag.load_state_dict(durum)
    return ag


def haritalar(ag, X):
    ag.eval()
    kayit = {}
    h = ag.blocks[3][2].register_forward_hook(lambda m, i, o: kayit.update(a=o))
    cikti = ag(torch.as_tensor(X))
    h.remove()
    g = torch.autograd.grad(cikti.sum(), kayit["a"])[0]
    cam = torch.relu((g.mean(dim=(2, 3), keepdim=True) * kayit["a"]).sum(dim=1, keepdim=True))
    return torch.nn.functional.interpolate(cam, size=(96, 96), mode="bilinear", align_corners=False)[:, 0].detach().numpy()

ag = egit()
haritalar_te = haritalar(ag, Xe[ete_])
maske = np.stack(d["sınır"])[ete_].astype(float)
pay = (haritalar_te * maske).sum(axis=(1, 2)) / np.maximum(haritalar_te.sum(axis=(1, 2)), 1e-12)
print("haritanın sınır içindeki payı", round(pay.mean(), 3), "| sınırın alanı", round(maske.mean(), 3))
cevap = round(float(pay.mean()), 3)
teach.check_step("Adım 2.4", "cevap", globals())

**Sonuç ne gösteriyor?** Bir haritanın ortalama 0,317 kadarı, görüntünün yaklaşık beşte birini kaplayan sınırların içine düşer: Haritalar kalbi tercih eder, ancak yalnızca kalbe bakmaktan çok uzaktır. Bunun modelle ilgili olup olmadığı, 7. bölümdeki ve haftalık projedeki sağlamlık testinin sorusudur.

**2. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#stop-2) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 3: Kayma, izlem ve alt gruplar

*Ders: [3. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#stop-3)* · *Python yolunda aynı sonuç: Kısım 2, 8-10. bölümler*

Bu durak bir modelin güvenilir kalıp kalmadığını sorar. 8. haftanın göğüs ağrısı ünitesinde bir sınıflandırıcı, henüz hiçbir sonuç bilinmeden, B hastanesinin başvurularını A hastanesininkilerden ayırmayı öğrenir. Ardından B'de gözlenen ve beklenen olaylar geliş sırasıyla izlenir. Risk modeli bir taburculuk eşiğinde cinsiyet ve yaş gruplarında değerlendirilir; koroner arter hastalığı modeli de cinsiyetle ve cinsiyet olmadan kurulur.

**Durağın sonunda:** Hastaneleri birbirinden ayıran sınıflandırıcının AUC değerini (0,988), B'de gözlenen eksi beklenen olayların bantlarının dışına ilk kez çıktığı başvuruyu (253), taburculuk eşiğinde kadınlardaki duyarlılığı (0,956) ve cinsiyet içermeyen bir modelin Cleveland verisindeki kadınlar için ortalama tahmin ettiği riski (0,417) bulacaksınız.

In [ ]:
#@title 3. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
_heart = load_cleveland()
# 2. haftanın ayırması, kendi sırasıyla: Önce eğitim, sonra test hastaları
_cad_tr, _cad_te = train_test_split(_heart.index.to_numpy(), test_size=0.25, stratify=_heart["disease"], random_state=RANDOM_STATE)
_CP = {1.0: "tipik anjina", 2.0: "atipik anjina", 3.0: "anjina dışı ağrı", 4.0: "ağrı yok"}


def _cad_table(order="split"):
    """Cleveland verisindeki 303 hasta; kümeleri, sekiz klinik değişken ve hastalıkla birlikte; 2. haftanın ayırmasının ya da hasta numaralarının sırasıyla."""
    h = _heart.loc[list(_cad_tr) + list(_cad_te)] if order == "split" else _heart
    return pd.DataFrame({"hasta": h.index, "küme": np.where(h.index.isin(_cad_tr), "eğitim", "test"),
                         "yaş": h["age"].astype(int), "cinsiyet": h["sex"].map({1.0: "erkek", 0.0: "kadın"}),
                         "istirahat kan basıncı": h["trestbps"].astype(int), "kolesterol": h["chol"].astype(int),
                         "en yüksek kalp hızı": h["thalach"].astype(int), "egzersiz anjinası": h["exang"].astype(int),
                         "ST çökmesi": h["oldpeak"], "göğüs ağrısı tipi": h["cp"].map(_CP),
                         "hastalık": h["disease"]}).reset_index(drop=True)



def _unit():
    """A ve B hastanelerinin başvuruları ile göğüs ağrısı ünitesinin risk modelinden MACE olasılıkları (bir kez hesaplanır, birkaç saniye sürer)."""
    if "_unit_cache" not in globals():
        def fit_logistic(X, y, Cs=(0.01, 0.1, 1.0, 10.0), Xv=None, yv=None):
            """Sütunları standartlaştırır ve bir lojistik regresyon kurar; doğrulama kümesi verilirse doğrulama AUC değeri en büyük olan ters düzenlileştirme gücünü tutar."""
            best = None
            for C in (Cs if Xv is not None else (1.0,)):
                model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)
                auc = roc_auc_score(yv, model.predict_proba(Xv)[:, 1]) if Xv is not None else 0.0
                if best is None or auc > best[1]:
                    best = (model, auc)
            return best[0]


        def logit(p):
            """Bir olasılığın log-odds değerini (logit) verir."""
            p = np.clip(p, 1e-6, 1 - 1e-6)
            return np.log(p / (1 - p))


        def bootstrap_auc(y, p, n=2000, seed=2026):
            """Hastalardan çekilen n bootstrap örneğinde AUC: n değerden oluşan bir dizi."""
            rng = np.random.default_rng(seed)
            out = []
            while len(out) < n:
                i = rng.integers(0, len(y), len(y))
                if 0 < y[i].sum() < len(i):
                    out.append(roc_auc_score(y[i], p[i]))
            return np.array(out)


        def paired_bootstrap(y, p1, p2, n=2000, seed=2026):
            """Aynı hastalardan çekilen n bootstrap örneğinde p1'in AUC değeri eksi p2'nin AUC değeri: n farktan oluşan bir dizi."""
            rng = np.random.default_rng(seed)
            out = []
            while len(out) < n:
                i = rng.integers(0, len(y), len(y))
                if 0 < y[i].sum() < len(i):
                    out.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
            return np.array(out)


        def calibration(y, p):
            """Genel kalibrasyon (eğim 1'e sabitlenmişken kesişim) ve kalibrasyon eğimi."""
            z = logit(p)
            a = 0.0
            for _ in range(50):
                q = 1 / (1 + np.exp(-(a + z)))
                a += (y - q).sum() / (q * (1 - q)).sum()
            slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], y).coef_[0, 0]
            return a, slope


        def reliability(y, p, bins=10):
            """Eşit büyüklükte gruplarda ortalama tahmin edilen risk ve gözlenen olay oranı: İki dizi."""
            order = np.argsort(p)
            groups = np.array_split(order, bins)
            return np.array([p[g].mean() for g in groups]), np.array([y[g].mean() for g in groups])


        def share_threshold(p, k):
            """Tahmin edilen riski en düşük k hastayı eve gönderen eşik: En düşük k. risk."""
            return float(np.sort(p)[k - 1])


        def rule_out_threshold(y, p, sensitivity):
            """Olaylar için duyarlılığı en az verilen düzeyde tutan en büyük eşik. Bu eşikte ya da altında kalan hastalar taburcu edilir. Eşik doğrulama kümesinde seçilir."""
            order = np.argsort(p)
            missed = np.cumsum(y[order])              # her eşikte ya da altında kalan olaylar
            ok = np.where(missed <= (1 - sensitivity) * y.sum())[0]
            return float(p[order][ok.max()]) if len(ok) else 0.0


        def wilson(k, n, z=1.96):
            """Bir k/n oranının yüzde 95 Wilson aralığı."""
            if n == 0:
                return (0.0, 1.0)
            p = k / n
            c = (p + z * z / (2 * n)) / (1 + z * z / n)
            h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
            return (c - h, c + h)


        def net_benefit(y, p, thresholds):
            """Her eşiğin üstündeki hastaları tedavi etmenin net yararı: Hasta başına doğru pozitifler eksi eşiğin olabilirliğiyle ağırlıklandırılmış yanlış pozitifler."""
            n = len(y)
            return np.array([((p >= t) & (y == 1)).sum() / n - ((p >= t) & (y == 0)).sum() / n * t / (1 - t) for t in thresholds])

        chest = pd.concat([triage.presentations("A"), triage.presentations("B")], ignore_index=True)
        A = chest[chest["hospital"] == "A"].reset_index(drop=True)
        B = chest[chest["hospital"] == "B"].reset_index(drop=True)
        tr, va, te = [(A["set"] == s).to_numpy() for s in ("training", "validation", "test")]
        y, yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()

        CLINICAL = ["age", "male", "smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family_history", "known_cad",
                    "heart_rate", "sbp", "egfr"]


        def clinical(d):
            """Klinik tablo: Yaş, cinsiyet, risk faktörleri, bilinen koroner arter hastalığı, kalp hızı, sistolik basınç ve eGFR."""
            return d.assign(male=(d["sex"] == "M").astype(int))[CLINICAL].astype(float)


        def laboratory(d):
            """Kendi testinin üst referans sınırının katları olarak troponin: Gelişteki ve bir saat sonraki değerin logaritması, bir saatteki değişim ve ikinci değerin olup olmadığı."""
            url = d["troponin_url"]
            t0, t1 = d["troponin_0h"] / url, d["troponin_1h"] / url
            return pd.DataFrame({"log_trop_0h": np.log(t0), "log_trop_1h": np.log(t1.fillna(t0)),
                                 "delta_1h": (t1 - t0).fillna(0.0), "has_1h": t1.notna().astype(float)})


        def echo(d):
            """Yatak başı ekokardiyogram: Yapılıp yapılmadığı, ejeksiyon fraksiyonu (yapılmadıysa 60) ve bölgesel duvar hareket bozukluğu (yapılmadıysa 0)."""
            return pd.DataFrame({"echo_done": d["echo_done"].astype(float), "lvef": d["lvef"].fillna(60.0), "rwma": d["rwma"].fillna(0.0)})

        tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(A.loc[tr, "note_tr"])
        svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(A.loc[tr, "note_tr"]))


        def note(d):
            """Triyaj notu 20 sayı olarak: Kelimelerin ve kelime çiftlerinin TF-IDF değerleri, eğitim notlarında öğrenilen bir tekil değer ayrışımıyla küçültülür."""
            return pd.DataFrame(svd.transform(tfidf.transform(d["note_tr"])), columns=[f"note_{i + 1}" for i in range(20)])

        inputs = lambda d: pd.concat([clinical(d), laboratory(d), echo(d), note(d)], axis=1)
        XA, XB = inputs(A), inputs(B)
        unit_model = fit_logistic(XA[tr], y[tr], Xv=XA[va], yv=y[va])
        pA, pB = unit_model.predict_proba(XA)[:, 1], unit_model.predict_proba(XB)[:, 1]
        globals()["_unit_cache"] = dict(A=A, B=B, pA=pA, pB=pB)
    return _unit_cache


_A, _B = _unit()["A"], _unit()["B"]
_va, _te = [(_A["set"] == _s).to_numpy() for _s in ("validation", "test")]
_SETS_U = {"training": "eğitim", "validation": "doğrulama", "test": "test"}


def _inputs_table():
    """A ve B hastanelerinin başvuruları; ekibin geliş anında bildikleriyle birlikte."""
    rows = []
    for name, d in (("A", _A), ("B", _B)):
        url = d["troponin_url"]
        rows.append(pd.DataFrame({"hastane": name, "küme": d["set"].map(_SETS_U).fillna("B hastanesi") if name == "A" else "B hastanesi",
                                  "yaş": d["age"], "cinsiyet": d["sex"].map({"M": "erkek", "F": "kadın"}),
                                  "sigara": d["smoker"], "diyabet": d["diabetes"], "hipertansiyon": d["hypertension"],
                                  "hiperlipidemi": d["hyperlipidaemia"], "obezite": d["obesity"],
                                  "aile öyküsü": d["family_history"], "bilinen koroner arter hastalığı": d["known_cad"],
                                  "kalp hızı": d["heart_rate"], "sistolik basınç": d["sbp"], "eGFR": d["egfr"],
                                  "gelişte troponin": (d["troponin_0h"] / url).round(2),
                                  "bir saat sonra troponin": (d["troponin_1h"] / url).round(2),
                                  "ekokardiyogram yapıldı": d["echo_done"], "ejeksiyon fraksiyonu": d["lvef"],
                                  "duvar hareket bozukluğu": d["rwma"], "triyaj notu": d["note_tr"]}))
    return pd.concat(rows, ignore_index=True)


print(len(_A), "başvuru A hastanesinde,", len(_B), "başvuru B hastanesinde")
print("3. durak hazır.")

### Adım 3.1: İki hastane birbirinden ayrılabiliyor mu?

**Amaç.** Bu adımda B hastanesinin başvurularını A hastanesininkilerden, hiçbir sonuç kullanmadan, girdilerine bakarak ayıran bir sınıflandırıcı eğitilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: A hastanesinin göğüs ağrısı ünitesi için kurulan bir risk modeli B hastanesinde kullanılacak. Kurul, ilk sonuçlar gelmeden önce B'nin hastalarının ve kayıtlarının A'nınkilere benzeyip benzemediğini ve nerede ayrıldıklarını öğrenmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda sanal kliniğin iki hastanesinden, yani sentetik bir veri setinden göğüs ağrısıyla yapılmış başvurular bulunur: A hastanesinde 4.000, B hastanesinde 1.000 başvuru. Her satır bir başvurudur: "hastane" hastaneyi ("A" ya da "B"), "küme" kümesini (A'da "eğitim", "doğrulama" ve "test"; B'de "B hastanesi") verir. Klinik tablo; "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), varsa 1 olan yedi risk faktörü ("sigara", "diyabet", "hipertansiyon", "hiperlipidemi", "obezite", "aile öyküsü" ve "bilinen koroner arter hastalığı"), "kalp hızı" (dakikada), "sistolik basınç" (mmHg) ve "eGFR" sütunlarını verir. Troponin, her hastanenin kendi testinin üst referans sınırının katları olarak verilir: "gelişte troponin" ve ölçülmediğinde boş kalan "bir saat sonra troponin". Yatak başı ekokardiyogram "ekokardiyogram yapıldı" (1 ya da 0) sütununu ve yapıldıysa "ejeksiyon fraksiyonu" (%) ile "duvar hareket bozukluğu" (bölgesel bir bozukluk görüldüyse 1) sütunlarını verir; yapılmadıysa bu iki sütun boştur. "triyaj notu" triyaj hemşiresinin Türkçe notudur.  
> Görev: Her başvuru için MACE risk modelinin girdilerini oluştur: Klinik tablo (yaş, erkek için 1 olan cinsiyet, yedi risk faktörü, kalp hızı, sistolik basınç, eGFR); troponin (gelişteki troponinin logaritması, ölçülmediyse gelişteki değerle bir saat sonraki troponinin logaritması, ölçülmediyse 0 ile bir saatteki değişim ve ikinci troponin ölçüldüyse 1); ekokardiyogram (ekokardiyogram yapıldı, yapılmadıysa 60 ile ejeksiyon fraksiyonu, yapılmadıysa 0 ile duvar hareket bozukluğu); ve 20 sayı olarak not (en az 3 notta geçen kelimelerin ve kelime çiftlerinin alt doğrusal terim sıklıklı TF-IDF değerleri; A'nın eğitim notlarında öğrenilir ve aynı notlarda öğrenilen bir tekil değer ayrışımıyla 2026 tohumuyla 20 bileşene indirilir). Ardından B hastanesini (1) A hastanesinden (0) ayıran, sütunları standartlaştırılmış ve en fazla 5.000 yinelemeli bir lojistik regresyon kur; tahminleri 5.000 başvurunun tamamında, 2026 tohumuyla karıştırılmış beş katlı tabakalı çapraz doğrulamadan al ve AUC değerini hesapla. Her girdi grubunun sütunlarıyla tek başına tekrarla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Bütün girdiler ve her grup için AUC değerini göster; bütün girdilerle elde edilen AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Hiçbir sonuç kullanılmamalı ve not modeli yalnızca A'nın eğitim notlarından öğrenmeli.  

In [ ]:
#@title Adım 3.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _inputs_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.1 kontrolü
teach.check_step("Adım 3.1", "cevap", globals())

In [ ]:
#@title Adım 3.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _inputs_table()
d = veri
t0, t1 = d["gelişte troponin"], d["bir saat sonra troponin"]
gruplar = {
    "klinik": pd.DataFrame({"yaş": d["yaş"], "erkek": (d["cinsiyet"] == "erkek").astype(int),
                                   **{s: d[s] for s in ["sigara", "diyabet", "hipertansiyon", "hiperlipidemi", "obezite",
                                                                  "aile öyküsü", "bilinen koroner arter hastalığı", "kalp hızı",
                                                                  "sistolik basınç", "eGFR"]}}),
    "troponin": pd.DataFrame({"log0": np.log(t0), "log1": np.log(t1.fillna(t0)), "degisim": (t1 - t0).fillna(0.0),
                              "ikinci": t1.notna().astype(int)}),
    "ekokardiyogram": pd.DataFrame({"yapildi": d["ekokardiyogram yapıldı"],
                                                    "ejeksiyon fraksiyonu": d["ejeksiyon fraksiyonu"].fillna(60.0),
                                                    "duvar hareketi": d["duvar hareket bozukluğu"].fillna(0.0)})}
egitim_notlari = d.loc[d["küme"] == "eğitim", "triyaj notu"]
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(egitim_notlari)
svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(egitim_notlari))
gruplar["not"] = pd.DataFrame(svd.transform(tfidf.transform(d["triyaj notu"])))
hastane = (d["hastane"] == "B").astype(int).to_numpy()
katlar = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)


def auc(X):
    p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)), X, hastane, cv=katlar, method="predict_proba")[:, 1]
    return roc_auc_score(hastane, p)


hepsi = pd.concat(list(gruplar.values()), axis=1)
hepsi.columns = [str(s) for s in hepsi.columns]
print("bütün girdiler", round(auc(hepsi), 3))
for ad, X in gruplar.items():
    print(ad, round(auc(X), 3))
cevap = round(float(auc(hepsi)), 3)
teach.check_step("Adım 3.1", "cevap", globals())

**Sonuç ne gösteriyor?** İki hastane 0,988 AUC değeriyle, neredeyse tamamen notlar sayesinde ayrılır: B'nin hemşireleri daha kısa notlar yazar. Sınıflandırıcı girdilerde neyin değiştiğini gösterir; risk modelinin hâlâ çalışıp çalışmadığını göstermez.

### Adım 3.2: B hastanesinde gözlenen ve beklenen olaylar

**Amaç.** Bu adımda B hastanesinde gözlenen eksi beklenen olaylar geliş sırasıyla izlenir ve ilk alarm bulunur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: A hastanesinin modeli artık B hastanesinde kullanılıyor ve hastaların sonuçları birbiri ardına geliyor. Kurul, model gereğinden az ya da fazla olay beklediğinde alarm veren basit bir grafik istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda sanal kliniğin ikinci hastanesi olan B hastanesinden, yani sentetik bir veri setinden göğüs ağrısıyla yapılmış 1.000 başvuru, geliş sırasıyla bulunur. Her satır bir başvurudur: "sıra" zamandaki yerini, "olasılık" birinci hastane olan A'da öğrenmiş bir modelin 30 gün içinde majör olumsuz kardiyak olay için verdiği olasılığı verir; "MACE" böyle bir olay görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Başvuruları sıralarıyla ilerle. Her m başvurusundan sonra m'ye kadarki MACE toplamı eksi olasılıkların toplamını ve m'ye kadarki p(1 − p) toplamının karekökünün 1,96 katı olan bandı hesapla. Farkın bandın üstünde kaldığı ilk başvuruyu bul; farkı ve bandı sıraya karşı çiz.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Grafiği göster, bütün başvurulardan sonraki farkı ve bandı yazdır; bandın üstündeki ilk başvurunun sırasını "cevap" adıyla sakla.  
> Test: Bant 0'a yakın başlamalı ve başvuru sayısıyla genişlemeli.  

In [ ]:
#@title Adım 3.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = pd.DataFrame({"sıra": np.arange(1, len(_B) + 1), "olasılık": _unit()["pB"], "MACE": _B["mace30"].to_numpy()})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.2 kontrolü
teach.check_step("Adım 3.2", "cevap", globals())

In [ ]:
#@title Adım 3.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = pd.DataFrame({"sıra": np.arange(1, len(_B) + 1), "olasılık": _unit()["pB"], "MACE": _B["mace30"].to_numpy()})
d = veri.sort_values("sıra")
p, y = d["olasılık"].to_numpy(), d["MACE"].to_numpy()
fark = np.cumsum(y - p)
bant = 1.96 * np.sqrt(np.cumsum(p * (1 - p)))
ust = np.where(fark > bant)[0]
fig, ax = plt.subplots(figsize=(6.4, 3))
ax.plot(d["sıra"], fark, color="#9B1B1B")
ax.fill_between(d["sıra"], -bant, bant, color="#1F5F8B", alpha=0.15)
ax.set_xlabel("geliş sırası")
ax.set_ylabel("gözlenen − beklenen")
plt.show()
print("sonunda:", round(fark[-1], 1), "bant ±", round(bant[-1], 1))
cevap = int(d["sıra"].to_numpy()[ust[0]]) if len(ust) else 0
teach.check_step("Adım 3.2", "cevap", globals())

**Sonuç ne gösteriyor?** Eğri bandının dışına ilk kez 253. başvuruda çıkar ve 1.000 başvurunun sonunda bandın çok üzerindedir: A'nın modeli B'de gereğinden az olay beklemektedir.

### Adım 3.3: Bir taburculuk eşiğinde alt gruplar

**Amaç.** Bu adımda doğrulama başvurularında bir taburculuk eşiği seçilir ve eşik test başvurularında cinsiyete ve yaşa göre değerlendirilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kurul, ortalamada güvenli olan bir taburculuk eşiğinin kadınlar, erkekler, 65 yaş altındaki ve 65 yaş ve üstündeki hastalar için de güvenli olup olmadığını öğrenmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda sanal kliniğin bir göğüs ağrısı ünitesinden, yani sentetik bir veri setinden 800 doğrulama ve 800 test başvurusu bulunur. Her satır bir başvurudur: "küme" kümesini ("doğrulama" ya da "test"), "cinsiyet" cinsiyeti ("erkek" ya da "kadın"), "yaş" yaşı (yıl), "olasılık" dersin eğitim başvurularıyla öğrenmiş bir modelinin 30 gün içinde majör olumsuz kardiyak olay için verdiği olasılığı verir; "MACE" böyle bir olay görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Eşiği yalnızca doğrulama başvurularında seç: MACE görülen doğrulama başvurularının en fazla yüzde 2'sinin eşikte ya da altında kaldığı en büyük olasılık. Eşikte ya da altında kalan başvurular eve gönderilir. Test başvurularında hepsi, kadınlar, erkekler, 65 yaş altındaki ve 65 yaş ve üstündeki hastalar için başvuru ve MACE sayısını, ROC eğrisi altındaki alanı (AUC), ortalama olasılığı, gözlenen MACE oranını, eve gönderilenlerin sayısını, bunlardaki MACE sayısını ve duyarlılığı hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her grup için bir satırdan oluşan bir tablo göster; kadınlardaki duyarlılığı üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Eşik yalnızca doğrulama başvurularından gelmeli.  

In [ ]:
#@title Adım 3.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_k = _va | _te
veri = pd.DataFrame({"küme": np.where(_va, "doğrulama", "test")[_k], "cinsiyet": _A["sex"].map({"M": "erkek", "F": "kadın"}).to_numpy()[_k],
                     "yaş": _A["age"].to_numpy()[_k], "olasılık": _unit()["pA"][_k], "MACE": _A["mace30"].to_numpy()[_k]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.3 kontrolü
teach.check_step("Adım 3.3", "cevap", globals())

In [ ]:
#@title Adım 3.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_k = _va | _te
veri = pd.DataFrame({"küme": np.where(_va, "doğrulama", "test")[_k], "cinsiyet": _A["sex"].map({"M": "erkek", "F": "kadın"}).to_numpy()[_k],
                     "yaş": _A["age"].to_numpy()[_k], "olasılık": _unit()["pA"][_k], "MACE": _A["mace30"].to_numpy()[_k]})
d = veri
do_, te_ = (d["küme"] == "doğrulama").to_numpy(), (d["küme"] == "test").to_numpy()
p, y = d["olasılık"].to_numpy(), d["MACE"].to_numpy()
sira = np.argsort(p[do_])
kacan = np.cumsum(y[do_][sira])
uygun = np.where(kacan <= 0.02 * y[do_].sum())[0]
esik = float(p[do_][sira][uygun.max()])
gruplar = {"hepsi": np.ones(len(d), bool), "kadınlar": (d["cinsiyet"] == "kadın").to_numpy(),
                  "erkekler": (d["cinsiyet"] == "erkek").to_numpy(), "65 yaş altı": (d["yaş"] < 65).to_numpy(),
                  "65 yaş ve üstü": (d["yaş"] >= 65).to_numpy()}
satirlar = []
for ad, g in gruplar.items():
    m = g & te_
    ev = p[m] <= esik
    satirlar.append({"grup": ad, "n": int(m.sum()), "MACE": int(y[m].sum()), "AUC": round(roc_auc_score(y[m], p[m]), 3),
                         "ortalama olasılık": round(p[m].mean(), 3), "gözlenen": round(y[m].mean(), 3),
                         "eve giden": int(ev.sum()), "bunlarda MACE": int(y[m][ev].sum()),
                         "duyarlılık": round(1 - y[m][ev].sum() / y[m].sum(), 3)})
tablo = pd.DataFrame(satirlar)
print("eşik", round(esik, 4))
print(tablo.to_string(index=False))
cevap = float(tablo.iloc[1]["duyarlılık"])
teach.check_step("Adım 3.3", "cevap", globals())

**Sonuç ne gösteriyor?** Eşik doğrulama başvurularında yüzde 98 duyarlılığı korur; ancak test kümesindeki kadınlar 0,956 düzeyinde kalır ve kaçırılan iki olayın ikisi de genç kadınlardadır. Her grupta az olay bulunduğu için kurul bu farkın kanıt olarak alınmasını değil, izlenmesini ister.

### Adım 3.4: Cinsiyet içermeyen bir model

**Amaç.** Bu adımda 2. haftanın lojistik regresyonu cinsiyetle ve cinsiyet olmadan kurulur ve kalibrasyonu ile hataları kadınlarda ve erkeklerde karşılaştırılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Biri, kadınlara ve erkeklere aynı biçimde davransın diye cinsiyetin koroner arter hastalığı modelinden çıkarılmasını öneriyor. Kurul bunun kadınlar ve erkekler için tahminlere ne yapacağını öğrenmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda UCI Heart Disease veri setinin Cleveland verisinden, yani koroner anjiyografiye gönderilmiş hastalardan 303 hasta bulunur. Her satır bir hastadır: "hasta" hastanın verideki numarasını (0-302), "küme" 2. haftanın kümesini verir ("eğitim", 227 hasta; "test", 76 hasta). Klinik değişkenler şunlardır: "yaş" (yıl), "cinsiyet" ("erkek" ya da "kadın"), "istirahat kan basıncı" (mmHg), "kolesterol" (mg/dL), "en yüksek kalp hızı" (egzersiz testinde ulaşılan en yüksek kalp hızı, dakikada), "egzersiz anjinası" (egzersiz testi anjinaya yol açtıysa 1), "ST çökmesi" (egzersizde ST segmentinin istirahate göre çökmesi, mm) ve "göğüs ağrısı tipi" ("tipik anjina", "atipik anjina", "anjina dışı ağrı" ya da "ağrı yok"). "hastalık" anjiyogramda koroner arter hastalığı görüldüyse 1, aksi hâlde 0 olur.  
> Görev: Tabloyu bu sırayla bir modelin sütunlarına dönüştür: Yaş, erkek için 1 ve kadın için 0 olan cinsiyet, istirahat kan basıncı, kolesterol, en yüksek kalp hızı, egzersiz anjinası, ST çökmesi ve atipik anjina, anjina dışı ağrı ve ağrı yok için 0 ve 1 değerli üç sütun. Satırları tablonun sırasında tut ve 303 hastanın tamamını kullan. Sütunları standartlaştırılmış ve en fazla 1.000 yinelemeli bir lojistik regresyonun 2026 tohumuyla karıştırılmış beş katlı tabakalı çapraz doğrulamasıyla her hasta için bir tahmin al; bunu bir kez cinsiyetle, bir kez de cinsiyet sütunu olmadan yap. Kadınlar ve erkekler için ortalama tahmin edilen riski gözlenen hastalık oranıyla karşılaştır; 0,5 olasılıkta duyarlılığı ve özgüllüğü hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her model ve cinsiyet için bir satırdan oluşan bir tablo göster; cinsiyet olmadan kadınlar için ortalama tahmin edilen riski üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her tahmin, hastayı içermeyen bir kattan gelmeli.  

In [ ]:
#@title Adım 3.4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _cad_table(order="number")
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.4 kontrolü
teach.check_step("Adım 3.4", "cevap", globals())

In [ ]:
#@title Adım 3.4 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _cad_table(order="number")
d = veri
agri = d["göğüs ağrısı tipi"]
X = pd.DataFrame({"yaş": d["yaş"], "erkek": (d["cinsiyet"] == "erkek").astype(int),
                    "istirahat kan basıncı": d["istirahat kan basıncı"], "kolesterol": d["kolesterol"],
                    "en yüksek kalp hızı": d["en yüksek kalp hızı"], "egzersiz anjinası": d["egzersiz anjinası"],
                    "ST çökmesi": d["ST çökmesi"],
                    "atipik anjina": (agri == "atipik anjina").astype(int),
                    "anjina dışı ağrı": (agri == "anjina dışı ağrı").astype(int),
                    "ağrı yok": (agri == "ağrı yok").astype(int)})
y = d["hastalık"].to_numpy()
eg_ = (d["küme"] == "eğitim").to_numpy()
te_ = ~eg_
oyuncular = [[0], [1], [2], [3], [4], [5], [6], [7, 8, 9]]

katlar = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
kadin = (d["cinsiyet"] == "kadın").to_numpy()
satirlar = []
for ad, XX in (("cinsiyetli", X), ("cinsiyetsiz", X.drop(columns="erkek"))):
    p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), XX, y, cv=katlar, method="predict_proba")[:, 1]
    for cins, m in (("kadınlar", kadin), ("erkekler", ~kadin)):
        satirlar.append({"model": ad, "cinsiyet": cins, "ortalama tahmin": round(p[m].mean(), 3),
                             "gözlenen": round(y[m].mean(), 3),
                             "duyarlılık": round(((p[m] >= 0.5) & (y[m] == 1)).sum() / (y[m] == 1).sum(), 3),
                             "özgüllük": round(((p[m] < 0.5) & (y[m] == 0)).sum() / (y[m] == 0).sum(), 3)})
tablo = pd.DataFrame(satirlar)
print(tablo.to_string(index=False))
cevap = float(tablo.iloc[2]["ortalama tahmin"])
teach.check_step("Adım 3.4", "cevap", globals())

**Sonuç ne gösteriyor?** Cinsiyet olmadan model kadınlar için 0,417 öngörür; bu değer kadınlardaki hastalık oranının çok üstündedir. Model erkeklerin riskini ise olduğundan düşük tahmin eder. Değişkeni dışarıda bırakmak hataları ortadan kaldırmaz, cinsiyetler arasında yer değiştirir.

**3. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#stop-3) dönün ve duraktan sonrasını okumaya devam edin.

## Prompt yolunda haftalık proje

*Haftanın projesi: [9. haftanın README sayfası, Haftalık proje](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-09/README.md#haftal%C4%B1k-proje)*

Haftalık proje, her adım için bir prompt olmak üzere aşağıdaki promptlarla da yapılabilir. Önce hazırlık hücresini bir kez çalıştırın; hücre projenin verilerini yükler. Ardından her adım, duraklardaki adımlar gibi ilerler. Tek fark şudur: Projede kontrol hücresi ve hazır kod yoktur, çünkü sonuçlar rapora girer. Bir promptun testi sağlanmadan sonraki prompta geçilmez ve her prompt YZ kullanım günlüğüne yazılır.

In [ ]:
#@title Projenin hazırlığı: Bu hücreyi bir kez çalıştırın (projenin verilerini yükler)
_SETS_E = {"training": "eğitim", "validation": "doğrulama", "testing": "test"}


def _echo():
    """Dersin CAMUS görüntüleri; kareleri ve uzman sınırlarıyla birlikte (bir kez yüklenir, birkaç saniye sürer)."""
    if "_echo_cache" not in globals():
        frames_ = camus.frames(camus.patients(), size=96)
        info = camus.table()
        low_ef = dict(zip(info["patient"], (info["ef"] < 45).astype(int)))
        index = {key: i for i, key in enumerate(zip(frames_["patient"], frames_["view"], frames_["phase"]))}
        views = sorted(set(zip(frames_["patient"], frames_["view"])))     # 300 hasta x 2 görüntü
        echo_x = np.stack([[frames_["images"][index[(p, v, "ED")]], frames_["images"][index[(p, v, "ES")]]] for p, v in views]).astype(np.float32)
        labels = [np.isin(frames_["masks"][index[(p, v, phase)]], (1, 2)) for p, v in views for phase in ("ED", "ES")]
        outline = np.array(labels).reshape(len(views), 2, 96, 96).any(axis=1)   # ED ya da ES karesinde kavite ya da miyokard
        echo_y = np.array([low_ef[p] for p, v in views])
        echo_set = np.array([frames_["split"][index[(p, v, "ED")]] for p, v in views])
        etr, eva, ete = echo_set == "training", echo_set == "validation", echo_set == "testing"
        globals()["_echo_cache"] = dict(views=views, echo_x=echo_x, outline=outline, echo_y=echo_y, echo_set=echo_set,
                                        quality=camus.table().set_index("patient"))
    return _echo_cache


def _echo_table():
    """Dersin 300 CAMUS hastasının 600 görüntüsü; kareleri, uzman sınırı ve yüzde 45'in altındaki ejeksiyon fraksiyonuyla birlikte."""
    e = _echo()
    q = e["quality"]
    return pd.DataFrame({"hasta": [p for p, v in e["views"]], "görüntü": [v for p, v in e["views"]],
                         "küme": [_SETS_E[s] for s in e["echo_set"]],
                         "görüntü kalitesi": [q.loc[p, "quality_2ch" if v == "2CH" else "quality_4ch"] for p, v in e["views"]],
                         "diyastol sonu": list(e["echo_x"][:, 0]), "sistol sonu": list(e["echo_x"][:, 1]),
                         "sınır": list(e["outline"].astype(np.uint8)), "düşük ejeksiyon fraksiyonu": e["echo_y"]})



def _project_maps():
    """100 test görüntüsü; dersin eko ağının olasılığı ve Grad-CAM haritasıyla birlikte (bir kez hesaplanır, bir dakikadan kısa sürer)."""
    if "_maps_cache" in globals():
        return _maps_cache
    e = _echo()
    class EchoNet(nn.Module):
        """Tek bir görüntü için küçük bir evrişimli ağ: Diyastol sonu ve sistol sonu kareleri iki kanal olarak girer; ağ dört bloktan ve tek bir çıktıdan, yani yüzde 45'in altındaki bir ejeksiyon fraksiyonunun logitinden oluşur."""
        def __init__(self):
            super().__init__()
            sizes = [2, 16, 32, 64, 64]
            self.blocks = nn.Sequential(*[nn.Sequential(nn.Conv2d(sizes[i], sizes[i + 1], 3, padding=1), nn.BatchNorm2d(sizes[i + 1]), nn.ReLU(), nn.MaxPool2d(2))
                                          for i in range(4)])
            self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1))

        def forward(self, x):
            return self.head(self.blocks(x)).squeeze(1)


    def train_echo(model, X, y, Xv, yv, epochs=30, seed=2026, verbose=True):
        """32 görüntülük gruplarla, öğrenme hızı 0,001 olan Adam; her görüntü en fazla 4 piksel kaydırılır. Doğrulama AUC değeri en iyi olan dönemin ağırlıkları saklanır."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X), torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 32):
                b = order[i:i + 32]
                dy, dx = (int(s) for s in torch.randint(-4, 5, (2,)))
                loss = loss_fn(model(torch.roll(X[b], shifts=(dy, dx), dims=(2, 3))), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, echo_predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose and (epoch + 1) % 5 == 0:
                print(f"dönem {epoch + 1}: doğrulama AUC {a:.3f}")
        model.load_state_dict(state)
        return best


    @torch.no_grad()
    def echo_predict(model, X, batch=64):
        model.eval()
        return torch.cat([torch.sigmoid(model(torch.as_tensor(X[i:i + batch]))) for i in range(0, len(X), batch)]).numpy()


    def grad_cam(model, X, layer):
        """Bir grup görüntü için çıktının Grad-CAM haritaları: Bir katmanın öznitelik haritaları gradyanlarının ortalamasıyla ağırlıklandırılır, toplanır, 0'ın üstünde kalan kısmı tutulur ve girdinin boyutuna büyütülür."""
        model.eval()
        saved = {}
        hook = layer.register_forward_hook(lambda module, inputs, output: saved.update(maps=output))
        output = model(torch.as_tensor(X))
        hook.remove()
        gradients = torch.autograd.grad(output.sum(), saved["maps"])[0]
        alpha = gradients.mean(dim=(2, 3), keepdim=True)                       # kanal başına bir ağırlık
        cam = torch.relu((alpha * saved["maps"]).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=tuple(X.shape[-2:]), mode="bilinear", align_corners=False)[:, 0]
        return cam.detach().numpy()


    def share_inside(cam, mask):
        """Her haritanın ağırlığının sınırın içine düşen payı."""
        return (cam * mask).sum(axis=(1, 2)) / np.maximum(cam.sum(axis=(1, 2)), 1e-12)


    def peak_inside(cam, mask):
        """Haritanın en yüksek değerli pikseli sınırın içindeyse 1."""
        return np.array([m.flat[c.argmax()] for c, m in zip(cam, mask)]).astype(int)
    etr, eva, ete = [e["echo_set"] == s for s in ("training", "validation", "testing")]
    torch.manual_seed(RANDOM_STATE)
    net = EchoNet()
    train_echo(net, e["echo_x"][etr], e["echo_y"][etr], e["echo_x"][eva], e["echo_y"][eva], verbose=False)
    X = e["echo_x"][ete]
    t = _echo_table()[ete].reset_index(drop=True)
    t["olasılık"] = echo_predict(net, X).round(4)
    t["harita"] = list(grad_cam(net, X, net.blocks[3][2]))
    globals()["_maps_cache"] = t
    return t
print("Proje hazır.")

### Proje adımı 1: Eko ağının doğruluğu

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kurul, ekokardiyogramlarda düşük ejeksiyon fraksiyonunu işaretleyen bir ağı inceliyor ve haritalara bakmadan önce ağın tek tek görüntülerdeki ve hastalardaki doğruluğunu aralıklarıyla birlikte görmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda herkese açık bir ekokardiyogram veri seti olan CAMUS'tan 600 görüntü, yani 300 hastanın apikal iki boşluk ve dört boşluk görüntüleri bulunur. Her satır bir görüntüdür: "hasta" hastayı, "görüntü" görüntüyü ("2CH" ya da "4CH"), "küme" hastanın kümesini ("eğitim", 200 hasta; "doğrulama", 50 hasta; "test", 50 hasta), "görüntü kalitesi" ise uzmanların verdiği derecesini ("Good", "Medium" ya da "Poor") verir. "diyastol sonu" ve "sistol sonu" sütunlarının her hücresi kareyi 0 ile 1 arasında değerler alan 96 × 96 piksellik bir görüntü olarak tutar. "sınır" sütununun her hücresinde ise sol ventrikül kavitesinin ve miyokardın diyastol sonu ya da sistol sonundaki uzman sınırı aynı boyutta yer alır: İçeride 1, dışarıda 0. "düşük ejeksiyon fraksiyonu" hastanın ejeksiyon fraksiyonu yüzde 45'in altındaysa 1, aksi hâlde 0 olur.  
> Görev: Tek bir görüntü için diyastol sonu ve sistol sonu karelerini 96 × 96 piksellik iki kanal olarak alan küçük bir evrişimli ağ kur. Ağda dört blok bulunsun; her blok, boyutu koruyan 3 × 3 çekirdekli bir evrişimden (sırasıyla 16, 32, 64 ve 64 kanal), yığın normalleştirmesinden, ReLU'dan ve 2 × 2 en büyük değer havuzlamasından oluşsun. Bloklardan sonra ağ her kanalın görüntü üzerindeki ortalamasını alsın ve 0,3 oranında seyreltme uygulasın; tek çıktılı bir doğrusal katman da düşük ejeksiyon fraksiyonunun logitini versin. Ağı eğitim görüntülerinde Adam (öğrenme hızı 0,001, ağırlık azaltma 0,0001) ve ikili çapraz entropiyle, 32 görüntülük karıştırılmış gruplarla 30 dönem eğit. Her grubu her yönde en fazla 4 piksel rastgele kaydır ve doğrulama görüntülerinde AUC değeri en iyi olan dönemin ağırlıklarını sakla. Ağ kütüphanesinin rastgele sayı üretecini ağı kurmadan önce ve eğitimden önce yeniden 2026 ile başlat. AUC değerini test görüntülerinde ve her hastanın iki görüntüsünün olasılıklarının ortalamasıyla test hastalarında hesapla. Her AUC değerine test görüntülerinden ya da test hastalarından çekilen 2.000 bootstrap örneğinden, 2,5. ve 97,5. yüzdelikler arasında bir yüzde 95 aralığı ver. Sonucun tekrarlanabilmesi için rastgelelik tohumunu 2026 olarak sabitle.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki AUC değerini aralıklarıyla birlikte bir tabloda göster; görüntü düzeyindeki test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Dönemi seçmek için hiçbir test görüntüsü kullanılmamalı ve hastaların bootstrap örnekleri görüntüleri değil, hastaları çekmeli.  

In [ ]:
#@title Proje adımı 1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _echo_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 2: Uzman sınırına karşı haritalar

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kurul, ağın düşük ejeksiyon fraksiyonunu işaretlerken sol ventriküle bakıp bakmadığını ve bunun ejeksiyon fraksiyonu düşük ve normal olan görüntüler, doğru ve yanlış tahminler ve görüntü kalitesi dereceleri arasında değişip değişmediğini öğrenmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda herkese açık bir ekokardiyogram veri seti olan CAMUS'tan dersin 100 test görüntüsü, yani 50 hastanın apikal iki boşluk ve dört boşluk görüntüleri bulunur. Her satır bir görüntüdür: "hasta", "görüntü", "küme" ve "görüntü kalitesi" (uzmanların verdiği derece: "Good", "Medium" ya da "Poor") görüntüyü tanımlar. "diyastol sonu" ve "sistol sonu" kareleri 0 ile 1 arasında değerler alan 96 × 96 piksellik görüntüler olarak tutar; "sınır" sol ventrikül kavitesinin ve miyokardın uzman sınırını aynı boyutta, içeride 1 ve dışarıda 0 olarak tutar; "düşük ejeksiyon fraksiyonu" hastanın ejeksiyon fraksiyonu yüzde 45'in altındaysa 1 olur. İki sütun, dersin eğitim görüntüleriyle eğitilmiş eko ağından gelir: "olasılık" ağın düşük ejeksiyon fraksiyonu için verdiği olasılık, "harita" ise ağın son bloktaki Grad-CAM haritası, yani 0 ya da daha büyük 96 × 96 değerden oluşan bir görüntüdür.  
> Görev: Her görüntü için üç ölçüt hesapla: "harita" toplamının "sınır" içine düşen payı; "harita" sütununun en yüksek değerli pikseli "sınır" içindeyse 1, değilse 0; ve "sınır" sütununun görüntüde kapladığı pay. Bir tahmin, düşük ejeksiyon fraksiyonlu bir görüntüde "olasılık" en az 0,5 olduğunda ya da düşük ejeksiyon fraksiyonu olmayan bir görüntüde 0,5'in altında kaldığında doğrudur. Üç ölçütün ortalamalarını bütün görüntüler, ejeksiyon fraksiyonu düşük ve normal olan görüntüler, doğru ve yanlış tahminler ve her görüntü kalitesi derecesi için, her gruptaki görüntü sayısıyla birlikte göster.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Grupların tablosunu göster; bütün görüntülerde sınır içindeki ortalama payı üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her pay 0 ile 1 arasında olmalı ve her karşılaştırmadaki grupların toplamı 100 görüntü etmeli.  

In [ ]:
#@title Proje adımı 2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _project_maps()   # ilk seferde bir dakikadan kısa sürer
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 3: Haritaların sağlamlık testleri

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kurul, haritaların ağın öğrendiklerine bağlı olup olmadığını öğrenmek istiyor. Ağın ağırlıkları rastgele değerlerle değiştirildiğinde aynı kalan bir harita modeli değil, görüntüyü anlatır; tek bir rastgele çekiliş de yanıltabilir.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda herkese açık bir ekokardiyogram veri seti olan CAMUS'tan 600 görüntü, yani 300 hastanın apikal iki boşluk ve dört boşluk görüntüleri bulunur. Her satır bir görüntüdür: "hasta" hastayı, "görüntü" görüntüyü ("2CH" ya da "4CH"), "küme" hastanın kümesini ("eğitim", 200 hasta; "doğrulama", 50 hasta; "test", 50 hasta), "görüntü kalitesi" ise uzmanların verdiği derecesini ("Good", "Medium" ya da "Poor") verir. "diyastol sonu" ve "sistol sonu" sütunlarının her hücresi kareyi 0 ile 1 arasında değerler alan 96 × 96 piksellik bir görüntü olarak tutar. "sınır" sütununun her hücresinde ise sol ventrikül kavitesinin ve miyokardın diyastol sonu ya da sistol sonundaki uzman sınırı aynı boyutta yer alır: İçeride 1, dışarıda 0. "düşük ejeksiyon fraksiyonu" hastanın ejeksiyon fraksiyonu yüzde 45'in altındaysa 1, aksi hâlde 0 olur.  
> Görev: Tek bir görüntü için diyastol sonu ve sistol sonu karelerini 96 × 96 piksellik iki kanal olarak alan küçük bir evrişimli ağ kur. Ağda dört blok bulunsun; her blok, boyutu koruyan 3 × 3 çekirdekli bir evrişimden (sırasıyla 16, 32, 64 ve 64 kanal), yığın normalleştirmesinden, ReLU'dan ve 2 × 2 en büyük değer havuzlamasından oluşsun. Bloklardan sonra ağ her kanalın görüntü üzerindeki ortalamasını alsın ve 0,3 oranında seyreltme uygulasın; tek çıktılı bir doğrusal katman da düşük ejeksiyon fraksiyonunun logitini versin. Ağı eğitim görüntülerinde Adam (öğrenme hızı 0,001, ağırlık azaltma 0,0001) ve ikili çapraz entropiyle, 32 görüntülük karıştırılmış gruplarla 30 dönem eğit. Her grubu her yönde en fazla 4 piksel rastgele kaydır ve doğrulama görüntülerinde AUC değeri en iyi olan dönemin ağırlıklarını sakla. Ağ kütüphanesinin rastgele sayı üretecini ağı kurmadan önce ve eğitimden önce yeniden 2026 ile başlat. Her test görüntüsü için çıktının Grad-CAM haritasını son bloğun ReLU katmanında, havuzlamadan önce hesapla. Bunun için çıktının bu katmandaki 12 × 12 boyutlu 64 öznitelik haritasına göre gradyanını al, her kanalın ortalama gradyanını o kanalın ağırlığı olarak kullan ve 64 haritanın ağırlıklı toplamını oluştur. Toplamın yalnızca pozitif kısmını tut ve çift doğrusal aradeğerlemeyle 96 × 96 boyutuna büyüt. Her görüntü için haritanın toplamının sınırın içine düşen payını hesapla. Ardından 1'den 20'ye kadar her tohum için şunu iki kez tekrarla: Bir kez son bloğuna ve çıktı katmanına yeni rastgele ağırlıklar verilen bir kopya için, bir kez de bütün katmanlarına yeni rastgele ağırlıklar verilen bir kopya için. Yeni ağırlıklar çekilmeden hemen önce ağ kütüphanesinin rastgele sayı üretecini o tohumla başlat ve eğitilmiş ağı değiştirmeden tut. Kopya için aynı haritaları hesapla; her görüntü için eğitilmiş ağın haritasının 9.216 pikseli ile kopyanın haritasının pikselleri arasındaki Spearman sıra korelasyonunu ve kopyanın haritasının sınır içindeki payını hesapla. İki haritadan biri her pikselde aynı değeri alıyorsa o görüntüyü korelasyonun dışında bırak. Her kopya için ortalama sıra korelasyonunu, sınırın içindeki ortalama payı ve boş harita sayısını kaydet. Her kopya türü için ortalama sıra korelasyonunun ve sınırın içindeki ortalama payın 20 tohum üzerindeki ortancasını ve aralığını, haritaların yarısından fazlasının boş kaldığı tohum sayısını ve bunların yanında eğitilmiş ağın sınır içindeki ortalama payını bildir.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Sonuçları küçük bir tabloda göster; son bloğu rastgele olan kopyalar için ortalama sıra korelasyonunun ortancasını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her sıra korelasyonu −1 ile 1 arasında olmalı ve eğitilmiş ağ, kopyalar alınmadan önce ve sonra aynı haritaları vermeli.  

In [ ]:
#@title Proje adımı 3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _echo_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 4: Kurul için altı görüntü

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kurul, ağın yanıldığı görüntüler de dahil olmak üzere birkaç görüntünün haritalarını uzman sınırlarının yanında görmek istiyor.  
> Veri: Defterde "veri" adlı bir tablo vardır. Tabloda herkese açık bir ekokardiyogram veri seti olan CAMUS'tan dersin 100 test görüntüsü, yani 50 hastanın apikal iki boşluk ve dört boşluk görüntüleri bulunur. Her satır bir görüntüdür: "hasta", "görüntü", "küme" ve "görüntü kalitesi" (uzmanların verdiği derece: "Good", "Medium" ya da "Poor") görüntüyü tanımlar. "diyastol sonu" ve "sistol sonu" kareleri 0 ile 1 arasında değerler alan 96 × 96 piksellik görüntüler olarak tutar; "sınır" sol ventrikül kavitesinin ve miyokardın uzman sınırını aynı boyutta, içeride 1 ve dışarıda 0 olarak tutar; "düşük ejeksiyon fraksiyonu" hastanın ejeksiyon fraksiyonu yüzde 45'in altındaysa 1 olur. İki sütun, dersin eğitim görüntüleriyle eğitilmiş eko ağından gelir: "olasılık" ağın düşük ejeksiyon fraksiyonu için verdiği olasılık, "harita" ise ağın son bloktaki Grad-CAM haritası, yani 0 ya da daha büyük 96 × 96 değerden oluşan bir görüntüdür.  
> Görev: Altı görüntü seç: Düşük ejeksiyon fraksiyonlu ve "olasılık" değeri en az 0,5 olan iki görüntü, düşük ejeksiyon fraksiyonlu ve "olasılık" değeri 0,5'in altında olan bir görüntü, düşük ejeksiyon fraksiyonu olmayan ve "olasılık" değeri en az 0,5 olan bir görüntü, düşük ejeksiyon fraksiyonu olmayan ve "olasılık" değeri 0,5'in altında olan bir görüntü ve görüntü kalitesi "Poor" olanlar arasında "olasılık" değeri en yüksek görüntü. Bunları üçer panelli iki satırda çiz: Diyastol sonu karesi gri tonlarda, üstünde kısmen saydam ve sıcak bir renk ölçeğinde "harita", "sınır" ise bir kontur çizgisi olarak. Her panelin başlığına hastayı, görüntüyü, olasılığı, gerçek sınıfı ve haritanın sınır içindeki payını yaz; şeklin altına "CAMUS, CC BY-NC-SA 4.0" notunu ekle.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Şekli göster; altı panelin hasta ve görüntülerini bir liste olarak "cevap" adıyla sakla.  
> Test: Altı panelden en az ikisi yanlış bir tahmini göstermeli.  

In [ ]:
#@title Proje adımı 4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _project_maps()   # ilk seferde bir dakikadan kısa sürer
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

<a id="part-2"></a>

# Kısım 2: Python yolu

**Kimler için?** Bu kısım Python yazmayı öğrenmek isteyen katılımcılar içindir. Kısım 1'i çalışmış olmanız gerekmez: 0. bölümden sonra doğrudan buradan başlayın.

**Nasıl ilerler?** Hücreleri yukarıdan aşağıya Shift+Enter ile çalıştırın. Baştaki *Python adımı*, haftanın gerektirdiği dil bilgisini öğretir. Numaralı bölümlerdeki alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerde durur ve ancak denedikten sonra açılmalıdır. *Gözlemle ve düşün* hücreleri açık uçlu sorular sorar; bu soruların çözüm hücresi yoktur. *Python notu* hücreleri, yeni bir yapının ilk kullanıldığı yerde dili açıklar. *İleri düzey (isteğe bağlı)* olarak işaretlenen hücreler, programlama bilenler için ek çalışmalardır. Bu kısma ortadan başlayan katılımcı, o bölümün başındaki *Telafi* hücresini çalıştırır. Bazı alıştırma hücreleri `veri` adlı bir tabloyu hazırlayan bir satırla başlar. Bu tablo, etkileşimli labın prompt atölyesindeki alıştırma promptları için hazırlanmıştır; Python yolunda alıştırma, metninde adı geçen tablolarla çözülebilir.

## Python adımı 9: Bir modeli kodla açıklamak

Bu hafta bir tahmin hastanın değişkenleri arasında paylaştırılmaktadır ve her değişken grubu kendi tahminine ihtiyaç duyar. Bu adımda `itertools` modülünün her değişken grubunu ve değişkenlerin katılabileceği her sırayı nasıl listelediği, anahtarları demet olan bir sözlüğün her grubun değerini nasıl sakladığı, `math.factorial` fonksiyonunun Shapley formülünün ağırlıklarını nasıl verdiği, bir fonksiyonun başka bir fonksiyona değer olarak nasıl verildiği ve bir katkı grafiğinin katkıları iki renkle nasıl gösterdiği anlatılmaktadır. Sayılar 1. bölümdeki ormanın 172 numaralı hasta için verdiği değerlerdir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
players = ["göğüs ağrısı tipi", "egzersiz anjinası", "cinsiyet"]
for mask in itertools.product([0, 1], repeat=3):
    members = [name for name, m in zip(players, mask) if m == 1]
    print(mask, sum(mask), members)
print(len(list(itertools.product([0, 1], repeat=8))), "sekiz değişkenin koalisyonu")

**Python notu.** `itertools.product([0, 1], repeat=3)` üç kez 0 ya da 1 yazmanın bütün yollarını verir: `(0, 0, 0)` ile `(1, 1, 1)` arasındaki sekiz *demet* (tuple). Demet, yuvarlak parantez içinde yazılan ve değiştirilemeyen bir listedir. Her demet bir koalisyondur: 1, değişkenin hastanın değerini koruduğu anlamına gelir. `zip(players, mask)` her adı kendi 0 ya da 1 değeriyle eşleştirir; `[name for ... if m == 1]` biçimindeki *liste üreteci* (list comprehension) üyelerin adlarını tutar. `sum(mask)` üyeleri sayar. Sekiz değişkenle `list(...)` $2^8 = 256$ demet içerir.

In [ ]:
# Python alıştırması: anahtarları demet olan bir sözlük ve değişkenlerin sıraları
value = {(0, 0, 0): 0.552, (1, 0, 0): 0.660, (0, 1, 0): 0.617, (0, 0, 1): 0.456,
         (1, 1, 0): 0.799, (1, 0, 1): 0.584, (0, 1, 1): 0.533, (1, 1, 1): 0.727}
print("yalnızca göğüs ağrısı tipiyle değer:", value[(1, 0, 0)])
orders = list(itertools.permutations(range(3)))
print(len(orders), "sıra:", orders)
contribution = {j: [] for j in range(3)}
for order in orders:
    have = [0, 0, 0]
    for j in order:
        before = value[tuple(have)]
        have[j] = 1
        contribution[j].append(round(value[tuple(have)] - before, 3))
for j in range(3):
    print(f"{players[j]:20}", contribution[j], "ortalama", round(np.mean(contribution[j]), 3))

**Python notu.** Bir demet değiştirilemediği için bir sözlükte anahtar olarak kullanılabilir: `value[(1, 0, 0)]`, yalnızca göğüs ağrısı tipinin hastanın değerini koruduğu koalisyonun değeridir. `itertools.permutations(range(3))` 0, 1 ve 2 sayılarının altı sırasını listeler. Her sıra için `have` listesi boş başlar, katılan her değişken için bir 1 alır; `tuple(have)` de listeyi yeniden bir anahtara dönüştürür. Bir değişken katıldıktan sonraki ve önceki değerler arasındaki fark, o sıradaki katkısıdır; altı sıra üzerinden ortalaması da Shapley değeridir. Değerler, dersteki çözümlü örneğin değerleridir.

In [ ]:
# Python alıştırması: Formülün ağırlıkları ve başka bir fonksiyona verilen bir fonksiyon
def weight(size, n):
    return math.factorial(size) * math.factorial(n - size - 1) / math.factorial(n)


print("üç değişken için ağırlıklar:", [round(weight(s, 3), 3) for s in range(3)])
print("bir değişkeni içermeyen 128 koalisyon üzerinden toplam:",
      round(sum(weight(sum(m), 8) for m in itertools.product([0, 1], repeat=7)), 6))


def apply_to_all(function, values):
    return [function(v) for v in values]


print(apply_to_all(math.factorial, [0, 1, 2, 3, 8]))
print(apply_to_all(lambda s: round(weight(s, 8), 5), range(8)))

**Python notu.** `math.factorial(n)`, $n! = 1 \times 2 \times \ldots \times n$ değerini verir ve $0! = 1$ olur. `weight(size, n)` Shapley formülündeki $|S|!\,(n - |S| - 1)!/n!$ ağırlığıdır. Belirli bir değişkeni içermeyen 128 koalisyon üzerinden ağırlıkların toplamı 1'dir; bu yüzden Shapley değeri ağırlıklı bir ortalamadır. `apply_to_all` ilk argüman olarak bir *fonksiyon* alır ve onu her değer için çağırır: Fonksiyon da bir sayı gibi bir değerdir ve başka bir fonksiyona verilebilir. `lambda s: ...` adı olmayan küçük bir fonksiyonu tek satırda yazar. 2. bölümün kodu da bir modelin tahminini, koalisyonların değerlerini hesaplayan fonksiyona aynı yolla verir.

In [ ]:
# Python alıştırması: Riski yükselten katkıları kırmızı, düşürenleri mavi gösteren bir grafik
phi = {"göğüs ağrısı tipi": 0.152, "egzersiz anjinası": 0.105, "cinsiyet": -0.083}
names = sorted(phi, key=lambda k: abs(phi[k]))
fig, ax = plt.subplots(figsize=(5.6, 1.9))
ax.barh(names, [phi[k] for k in names], color=["#9B1B1B" if phi[k] > 0 else "#1F5F8B" for k in names])
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("Shapley değeri (olasılık)")
plt.show()

**Python notu.** `sorted(phi, key=...)` sözlüğün anahtarlarını bir kurala göre sıralar: Burada katkıların mutlak değerine göre; böylece en büyük çubuk sonda, yani grafiğin en üstünde yer alır. *Koşullu ifade* `"#9B1B1B" if phi[k] > 0 else "#1F5F8B"`, liste üretecinin içinde her çubuk için bir renk seçer. `ax.axvline(0)` katkı olmadığını gösteren çizgiyi çizer.

**Hızlı kontrol.** Sekiz değişkenin 256 koalisyonundan kaçı göğüs ağrısı tipi gibi belirli bir değişkeni içerir?

<details><summary>Cevap</summary>

128, yani yarısı: Öteki yedi değişkenin 128 koalisyonunun her biri için belirli değişken ya içeride ya dışarıdadır.

</details>

**Alıştırma 9.1.** Sekiz değişkenin 256 koalisyonundan kaçında tam olarak üç üye vardır? Sayıyı saklayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 9.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = sum(1 for m in itertools.product([0, 1], repeat=8) if sum(m) == 3)
teach.check("Alıştırma 9.1", cevap)

**Alıştırma 9.2.** Üç değişkenin `value` sözlüğüyle egzersiz anjinasının, yani 1 numaralı değişkenin Shapley değerini altı katkısının ortalaması olarak hesaplayın. Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("9.2")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 9.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(np.mean(contribution[1])), 3)
teach.check("Alıştırma 9.2", cevap)

**Alıştırma 9.3.** Sekiz değişken için Shapley formülünde üç üyeli bir koalisyonun ağırlığı nedir? Beş ondalık basamakla saklayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 9.3", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(weight(3, 8), 5)
teach.check("Alıştırma 9.3", cevap)

## 1. 2. haftanın koroner arter hastalığı modeli

*Ders: [Neden açıklamalı? Yorumlanabilir modeller ve sonradan yapılan açıklamalar](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#why-explain-interpretable-models-and-explanations-after-the-fact)*

Mini proje 2. haftanın Cleveland verisine döner: Koroner anjiyografiye gönderilmiş 303 hasta, sekiz klinik değişken ve anjiyogramdaki hastalık. 2. haftadakiyle aynı ayırma hastaların dörtte birini teste ayırır ve aynı iki model ötekilerle öğrenir: Bir lojistik regresyon ve bir rastgele orman. Haftanın vakası test kümesindeki 172 numaralı hastadır.

**Formüllerden koda.** $f$ modeli: `forest` ve `logistic`. $B$ arka plan hastaları: `X_train`. $x$ hastası: `X_test.loc[172]`. Sekiz oyuncu: `VARIABLES` sözlüğünün anahtarları; burada göğüs ağrısı tipinin üç sütunu vardır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 1. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB09_guvenilirlik_ve_aciklanabilir_yapay_zeka.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(1)

In [ ]:
heart = load_cleveland()
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
cad = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"], drop_first=True, dtype=int)
X_cad, y_cad = cad.drop(columns="disease"), cad["disease"]
X_train, X_test, y_train, y_test = train_test_split(X_cad, y_cad, test_size=0.25, stratify=y_cad, random_state=RANDOM_STATE)
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_train, y_train)
# sekiz klinik değişken; göğüs ağrısı tipinin üç sütunu tek bir değişken sayılır
VARIABLES = {"yaş": ["age"], "cinsiyet": ["sex"], "istirahat basıncı": ["trestbps"],
             "kolesterol": ["chol"], "en yüksek kalp hızı": ["thalach"],
             "egzersiz anjinası": ["exang"], "ST çökmesi": ["oldpeak"],
             "göğüs ağrısı tipi": ["cp_2.0", "cp_3.0", "cp_4.0"]}

print("eğitim", X_train.shape, "| test", X_test.shape, "| test kümesinde hastalık oranı", round(y_test.mean(), 3))
print("test AUC: orman", round(roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]), 3),
      "| lojistik regresyon", round(roc_auc_score(y_test, logistic.predict_proba(X_test)[:, 1]), 3))

**Python notu.** Bu, 2. haftanın kodudur ve iki eklemesi vardır. `cad`, 2. haftadaki `data` tablosunun yerini alır; böylece bu ad promptların tabloları için boş kalır. `VARIABLES`, her değişkenin klinik adından sütunlarının listesine giden bir sözlüktür: Yedi değişkenin tek bir sütunu vardır; göğüs ağrısı tipinin ise tipik anjinayı referans alan `cp_2.0`, `cp_3.0` ve `cp_4.0` kukla sütunları vardır. Bu haftanın açıklamaları bir tahmini bu sekiz değişken arasında paylaştırır, hiçbir zaman tek bir değişkenin üç sütunu arasında paylaştırmaz.

In [ ]:
CASE = 172
print(heart.loc[CASE, FEATURES + ["disease"]].to_string())
print("orman:", round(forest.predict_proba(X_test.loc[[CASE]])[0, 1], 3),
      "| lojistik regresyon:", round(logistic.predict_proba(X_test.loc[[CASE]])[0, 1], 3),
      "| eğitim hastalarında hastalık oranı:", round(y_train.mean(), 3))

**Python notu.** `heart.loc[CASE, FEATURES + ["disease"]]`, 172 etiketli satırı ve listelenen sütunları alır. İki çift köşeli parantezle yazılan `X_test.loc[[CASE]]` tek satırlı bir tablo tutar; `predict_proba` buna ihtiyaç duyar. Tek bir çift parantezle tek bir satır seri olarak gelirdi. 172 numaralı hasta, istirahatte göğüs ağrısı olmayan (`cp` 4) ve egzersiz testinde ST çökmesi olmadan (`oldpeak` 0) anjinası ortaya çıkan (`exang` 1) bir kadındır (`sex` 0).

**Gözlemle ve düşün.** İki model test hastalarını neredeyse aynı ölçüde iyi sıralamasına karşın orman 172 numaralı hastaya lojistik regresyondan çok daha yüksek bir olasılık verir. Herhangi bir açıklamayı okumadan önce, her modelin hastanın hangi değerlerini farklı tartmasını beklersiniz?

**Alıştırma A.** Orman 76 test hastasından kaçına en az 0,5 koroner arter hastalığı olasılığı verir? Sayıyı saklayın.

In [ ]:
veri = alistirma_tablosu("A")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma A", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int((forest.predict_proba(X_test)[:, 1] >= 0.5).sum())
teach.check("Alıştırma A", cevap)

## 2. Tek bir hastanın Shapley değerleri

*Ders: [Shapley değerleri: Bir tahmini değişkenler arasında paylaştırmak](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#shapley-values-sharing-a-prediction-among-the-variables)*

172 numaralı hasta için sekiz değişkenin 256 koalisyonunun hepsinin değerleri, 227 eğitim hastası arka plan alınarak hesaplanır; Shapley formülü de bunları sekiz değere dönüştürür. Lojistik regresyon için log-odds ölçeğindeki aynı hesaplama kapalı formla karşılaştırılır.

**Formüllerden koda.** Her $S$ koalisyonu için $v(S)$: `coalition_values(predict, patient, background)`; 0 ve 1'lerden oluşan bir demetten değere giden bir sözlük. $\phi_j$: `shapley(values)`. Olasılık ölçeğinde $f$ modeli: `risk_forest` ve `risk_logistic`; log-odds ölçeğinde: `logodds_logistic`. $\beta_j (x_j - \bar{x}_j)$ kapalı formu: `beta * (x - X_train.mean())`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 2. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB09_guvenilirlik_ve_aciklanabilir_yapay_zeka.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(2)

In [ ]:
def coalition_values(predict, patient, background, variables=VARIABLES):
    """Her S koalisyonunun değeri: S'deki değişkenler hastanın değerlerini korurken öteki değişkenler her arka plan hastasının değerlerini aldığında elde edilen ortalama tahmin. Her değişken için bir öğe içeren ve S'nin üyeleri için 1 olan, 0 ve 1'lerden oluşan bir demetten değere giden bir sözlük döndürür."""
    columns = list(background.columns)
    B = background.to_numpy(dtype=float)
    x = patient[columns].to_numpy(dtype=float)
    position = {name: [columns.index(c) for c in cols] for name, cols in variables.items()}
    masks = list(itertools.product([0, 1], repeat=len(variables)))
    rows = []
    for mask in masks:
        keep = np.zeros(len(columns), dtype=bool)
        for member, name in zip(mask, variables):
            if member:
                keep[position[name]] = True
        rows.append(np.where(keep, x, B))
    prediction = predict(pd.DataFrame(np.vstack(rows), columns=columns))
    means = prediction.reshape(len(masks), len(B)).mean(axis=1)
    return dict(zip(masks, means))


def shapley(values):
    """Bütün koalisyonların değerlerinden kesin Shapley değerleri: Her değişken için, onu içermeyen koalisyonlara kattığı değerin ağırlıklı ortalaması."""
    n = len(next(iter(values)))
    phi = np.zeros(n)
    for mask, value in values.items():
        size = sum(mask)
        for j in range(n):
            if mask[j] == 0:
                with_j = mask[:j] + (1,) + mask[j + 1:]
                weight = math.factorial(size) * math.factorial(n - size - 1) / math.factorial(n)
                phi[j] += weight * (values[with_j] - value)
    return phi


def risk_forest(d):
    return forest.predict_proba(d)[:, 1]


def risk_logistic(d):
    return logistic.predict_proba(d)[:, 1]


def logodds_logistic(d):
    return logistic.decision_function(d)

patient = X_test.loc[CASE]
values_forest = coalition_values(risk_forest, patient, X_train)
phi_forest = shapley(values_forest)
empty, full = (0,) * 8, (1,) * 8
print("başlangıç değeri v(∅):", round(values_forest[empty], 4), "| tahmin v(N):", round(values_forest[full], 4))
print(pd.Series(phi_forest, index=list(VARIABLES)).sort_values(key=abs, ascending=False).round(4))
print("değerlerin toplamı:", round(phi_forest.sum(), 6), "| v(N) − v(∅):", round(values_forest[full] - values_forest[empty], 6))

**Python notu.** `coalition_values` 256 koalisyonun her biri için bir tablo kurar: Bu tabloda üyelerin sütunları hastanın değerlerini, öteki sütunlar ise 227 eğitim hastasının değerlerini korur. `np.where(keep, x, B)`, satır satır, `keep` doğru olan yerlerde `x` dizisinden, öteki yerlerde `B` dizisinden değer alır. `np.vstack` 256 tabloyu 58.112 satırlık tek bir tabloda üst üste koyar; böylece model yalnızca bir kez tahmin yapar. `reshape(256, 227)` tahminleri koalisyon başına bir satır olacak biçimde yeniden düzenler ve `mean(axis=1)` her koalisyonun değerini verir. `dict(zip(masks, means))` demetleri ve değerleri bir sözlükte birleştirir. `shapley` içindeki `mask[:j] + (1,) + mask[j + 1:]`, $j$ değişkeni eklenmiş aynı koalisyonu kurar; `(1,)` tek öğeli bir demettir.

In [ ]:
scaler, lr = logistic[0], logistic[1]
beta = lr.coef_[0] / scaler.scale_                       # her sütunun birimi başına log-odds
terms = beta * (patient - X_train.mean())
closed = np.array([terms[cols].sum() for cols in VARIABLES.values()])
phi_logodds = shapley(coalition_values(logodds_logistic, patient, X_train))
phi_logistic = shapley(coalition_values(risk_logistic, patient, X_train))
print(pd.DataFrame({"kapalı form": closed, "tek tek hesap, log-odds": phi_logodds,
                    "tek tek hesap, olasılık": phi_logistic, "orman, olasılık": phi_forest},
                   index=list(VARIABLES)).round(4).to_string())
print("kapalı form ile tek tek hesap arasındaki en büyük fark:", float(np.abs(closed - phi_logodds).max()))
order = np.argsort(np.abs(phi_forest))
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.2), sharey=True)
for ax, phi, title in ((axes[0], phi_forest, "rastgele orman"), (axes[1], phi_logistic, "lojistik regresyon")):
    ax.barh(np.array(list(VARIABLES))[order], phi[order], color=np.where(phi[order] > 0, "#9B1B1B", "#1F5F8B"))
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Shapley değeri (olasılık)")
plt.tight_layout()
plt.show()

**Python notu.** `make_pipeline` ile kurulan bir işlem hattı bir liste gibi indekslenebilir: `logistic[0]` ölçekleyici, `logistic[1]` ise lojistik regresyondur. Regresyonun katsayıları standartlaştırılmış sütunlara aittir; bunları eğitim hastalarının standart sapmaları olan `scaler.scale_` değerlerine bölmek, her özgün sütunun birimi başına log-odds değişimini verir. `patient - X_train.mean()` ortalamaları sütun sütun çıkarır; çünkü ikisi de aynı sütun adlarıyla etiketlidir. `terms[cols].sum()` bir değişkenin sütunlarının terimlerini toplar. `np.argsort` mutlak değerleri sıralayacak konumları verir; böylece iki grafik de değişkenleri aynı sırayla listeler.

**Gözlemle ve düşün.** Orman ve lojistik regresyon 172 numaralı hasta için en çok hangi iki değişkende ayrışıyor? İstirahatte göğüs ağrısı olmayan 59 yaşındaki bir kadın için iki modelden hangisine güvenirdiniz ve karar vermeden önce neyi denetlerdiniz?

**Alıştırma B.** Lojistik regresyonun kapalı formuyla 172 numaralı hastanın istirahat kan basıncının log-odds ölçeğindeki Shapley değerini hesaplayın. Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("B")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma B", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(beta[2] * (patient["trestbps"] - X_train["trestbps"].mean())), 3)
teach.check("Alıştırma B", cevap)

## 3. Tek bir hastadan bütün hastalara

*Ders: [Tek bir hastadan bütün hastalara](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#from-one-patient-to-all-patients)*

Ormanın Shapley değerleri 76 test hastasının hepsi için yaklaşık yarım dakikada hesaplanır ve özetlenir: Her değişkenin mutlak değerlerinin ortalaması, her hastada başı çeken değişken ve her değerin değişkeninin değerine karşı çizildiği bir grafik. Sekiz değişken için hesaplanan 2. haftanın permütasyon önemi ikinci bir sıralama verir.

**Formüllerden koda.** Her test hastası için bir satır Shapley değeri: `S_forest`, 76 × 8 boyutlu bir dizi. Ortalama mutlak değer $\frac{1}{n}\sum_i |\phi_{ij}|$: `np.abs(S_forest).mean(axis=0)`. Bir değişkenin permütasyon önemi: AUC eksi sütunları birlikte karıştırıldıktan sonraki AUC.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 3. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB09_guvenilirlik_ve_aciklanabilir_yapay_zeka.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(3)

In [ ]:
S_forest = np.array([shapley(coalition_values(risk_forest, X_test.iloc[i], X_train)) for i in range(len(X_test))])
S_logistic = np.array([shapley(coalition_values(risk_logistic, X_test.iloc[i], X_train)) for i in range(len(X_test))])
summary = pd.DataFrame({"orman": np.abs(S_forest).mean(axis=0), "lojistik regresyon": np.abs(S_logistic).mean(axis=0),
                        "en büyük olduğu hasta (orman)": np.bincount(np.abs(S_forest).argmax(axis=1), minlength=8)},
                       index=list(VARIABLES))
print(summary.sort_values("orman", ascending=False).round(3).to_string())
rng = np.random.default_rng(RANDOM_STATE)
auc = roc_auc_score(y_test, risk_forest(X_test))
drop = {}
for name, cols in VARIABLES.items():
    losses = []
    for _ in range(20):
        shuffled = X_test.copy()
        shuffled[cols] = X_test[cols].to_numpy()[rng.permutation(len(X_test))]
        losses.append(auc - roc_auc_score(y_test, risk_forest(shuffled)))
    drop[name] = np.mean(losses)
print("permütasyon önemi (AUC'deki düşüş):", {k: round(float(v), 3) for k, v in drop.items()})
fig, ax = plt.subplots(figsize=(7.2, 3.6))
for k, name in enumerate(summary.sort_values("orman").index):
    j = list(VARIABLES).index(name)
    col = X_test[VARIABLES[name]].to_numpy() @ np.arange(1, len(VARIABLES[name]) + 1)
    level = (col - col.min()) / max(col.max() - col.min(), 1e-9)
    jitter = np.random.default_rng(j).uniform(-0.25, 0.25, len(col))
    ax.scatter(S_forest[:, j], k + jitter, c=level, cmap="coolwarm", s=14)
ax.set_yticks(range(8))
ax.set_yticklabels(summary.sort_values("orman").index)
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("ormanın Shapley değeri (olasılık); renk: değişkenin değeri, mavi düşük ve kırmızı yüksek")
plt.tight_layout()
plt.show()

**Python notu.** Liste üreteci her test hastası için `coalition_values` ve `shapley` fonksiyonlarını çağırır; `X_test.iloc[i]` hastayı konumuna göre alır. `np.bincount(..., minlength=8)` sekiz konumun her birinin kaç kez geçtiğini, burada her değişkenin kaç hastada en büyük mutlak değere sahip olduğunu sayar. Permütasyon öneminde `rng.permutation(len(X_test))` hastaların sırasını karıştırır; göğüs ağrısı tipinin üç sütunu tek bir değişken olarak birlikte karıştırılır. Grafikte her nokta bir hastadır: `k + jitter` bir değişkenin noktalarını dikey olarak yayar; `cmap="coolwarm"` da noktaları değişkenin değerine göre maviden kırmızıya boyar. Göğüs ağrısı tipinde renk tipik anjinadan ağrı yok seçeneğine doğru ilerler.

In [ ]:
# İsteğe bağlı: shap paketi kuruluysa lojistik regresyon için aynı değerler
try:
    import shap
    background = shap.maskers.Independent(scaler.transform(X_train), max_samples=len(X_train))
    values_shap = shap.LinearExplainer(lr, background).shap_values(scaler.transform(X_test.loc[[CASE]]))[0]
    grouped = [values_shap[[list(X_train.columns).index(c) for c in cols]].sum() for cols in VARIABLES.values()]
    print("shap", shap.__version__, np.round(grouped, 4))
    print("kapalı form", np.round(closed, 4))
except ImportError:
    print("shap paketi kurulu değil; yukarıdaki kapalı form aynı değerleri verir.")
except Exception as error:
    print("shap paketi burada kullanılamadı:", error)

**İleri düzey (isteğe bağlı).** Lundberg ve Lee'nin shap paketi birçok model türü için Shapley değerlerini hesaplar. Doğrusal bir model için kapalı formu kullanır; `max_samples=len(X_train)` bütün eğitim hastalarını arka plan olarak tutar. Paket varsayılan olarak bu hastalardan 100 tanesini çeker ve değerleri bu durumda tek tek hesaplananlardan biraz farklı olurdu. Paket göğüs ağrısı tipinin üç sütununu üç ayrı değişken olarak ele alır; bu yüzden kod bunları sonradan toplar. Log-odds ölçeğindeki doğrusal bir model için bu toplam tam olarak değişkenin değerini verir. Paketi orman üzerinde de denemek isterseniz Anaconda'da `pip install shap` komutuyla kurun.

**Gözlemle ve düşün.** Göğüs ağrısı tipi test hastalarının çoğunda ormanın başını çeker ve "ağrı yok" seçeneği tahmin edilen riski yükseltir. Bu, Cleveland verisindeki hastalar hakkında ne söyler ve aynı örüntüyü bir göğüs ağrısı ünitesinde de bekler miydiniz?

**Alıştırma C.** 76 test hastasının kaçında göğüs ağrısı tipi ormanda mutlak Shapley değeri en büyük olan değişkendir? Sayıyı saklayın.

In [ ]:
veri = alistirma_tablosu("C")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma C", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int((np.abs(S_forest).argmax(axis=1) == 7).sum())
teach.check("Alıştırma C", cevap)

## 4. 172 numaralı hastanın çevresinde LIME

*Ders: [LIME: Tek bir hastanın çevresinde basit bir model](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#lime-a-simple-model-around-one-patient)*

LIME tek bir tahmini, ormanı hastanın yakınında taklit eden basit bir modelle açıklar. Her komşu bazı değişkenlerde 172 numaralı hastanın değerini korur, ötekilerde rastgele bir eğitim hastasının değerlerini alır. Ormanın riskinin bu seçimler üzerine, her komşunun hastaya ne kadar yakın kaldığına göre ağırlıklandırılmış bir ridge regresyonu, her değişken için bir katsayı verir.

**Formüllerden koda.** İkili gösterim $z \in \{0, 1\}^8$: `keep`. Değiştirilen değişken sayısının karekökü olan $d(z)$ uzaklığı: `distance`. $w = 0{,}75\sqrt{8}$ ile $\pi(z) = \exp(-d(z)^2/w^2)$ çekirdeği: `weight`. Ağırlıklı doğrusal modelin katsayıları: `surrogate.coef_`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 4. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB09_guvenilirlik_ve_aciklanabilir_yapay_zeka.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(4)

In [ ]:
def lime(predict, patient, background, variables=VARIABLES, n=2000, width=None, seed=RANDOM_STATE):
    """Bir hasta için LIME: Her değişkenin hastanın değerini koruduğu ya da rastgele bir arka plan hastasının değerini aldığı n komşu, değiştirilen değişken sayısı arttıkça azalan ağırlıklar ve katsayıları açıklamayı oluşturan ağırlıklı bir doğrusal model."""
    rng = np.random.default_rng(seed)
    columns = list(background.columns)
    B = background.to_numpy(dtype=float)
    x = patient[columns].to_numpy(dtype=float)
    width = 0.75 * np.sqrt(len(variables)) if width is None else width
    keep = rng.integers(0, 2, size=(n, len(variables)))       # 1: değişken hastanın değerini korur
    rows = B[rng.integers(0, len(B), size=n)].copy()
    for j, cols in enumerate(variables.values()):
        idx = [columns.index(c) for c in cols]
        rows[:, idx] = np.where(keep[:, [j]] == 1, x[idx], rows[:, idx])
    risk = predict(pd.DataFrame(rows, columns=columns))
    distance = np.sqrt((1 - keep).sum(axis=1))                 # değiştirilen değişken sayısının karekökü
    weight = np.exp(-distance ** 2 / width ** 2)
    surrogate = Ridge(alpha=1.0).fit(keep, risk, sample_weight=weight)
    return surrogate.coef_

coef = lime(risk_forest, patient, X_train)
print(pd.DataFrame({"LIME": coef, "Shapley": phi_forest}, index=list(VARIABLES)).round(3))
print("LIME ile Shapley arasındaki korelasyon:", round(np.corrcoef(coef, phi_forest)[0, 1], 3))

**Python notu.** `np.random.default_rng(seed)` kendine ait bir rastgele sayı üreteci kurar; böylece komşular defterdeki başka bir rastgele çekilişe bağlı kalmaz. `rng.integers(0, 2, size=(n, 8))` her komşu için bir satır olmak üzere 0 ve 1'lerden oluşan bir tablo çeker; `B[rng.integers(0, len(B), size=n)]` de her komşu için bir eğitim hastası seçer. Döngünün içindeki `keep[:, [j]]` sütunu tek sütunlu bir tablo olarak tutar; böylece `np.where` onu $j$ değişkeninin sütunlarıyla satır satır karşılaştırabilir. `Ridge(alpha=1.0)` büyük katsayılara küçük bir ceza veren bir doğrusal regresyondur; `sample_weight=weight` ise yakın komşuların daha çok sayılmasını sağlar.

In [ ]:
runs = {n: np.array([lime(risk_forest, patient, X_train, n=n, seed=s) for s in range(1, 21)]) for n in (2000, 100)}
for n, r in runs.items():
    first = pd.Series([list(VARIABLES)[j] for j in np.abs(r).argmax(axis=1)]).value_counts()
    print(f"{n} komşu, 20 tohum: ilk değişken", {name: int(c) for name, c in first.items()})
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.2), sharey=True)
for ax, (n, r) in zip(axes, runs.items()):
    for j in range(8):
        ax.scatter(r[:, j], np.full(20, j) + np.random.default_rng(j).uniform(-0.2, 0.2, 20), color="#1F5F8B", s=10)
    ax.set_yticks(range(8))
    ax.set_yticklabels(list(VARIABLES))
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_title(f"{n} komşu, 20 tohum", fontsize=10)
    ax.set_xlabel("LIME katsayısı")
plt.tight_layout()
plt.show()

**Python notu.** *Sözlük üreteci* (dictionary comprehension) `{n: ... for n in (2000, 100)}`, her komşu sayısı için bir girdisi olan bir sözlük kurar; her girdi, her tohum için bir satır olmak üzere 20 satırlık bir dizidir. `np.abs(r).argmax(axis=1)` her satırda mutlak değeri en büyük katsayının konumunu bulur; `value_counts` de her değişkenin kaç kez ilk sıraya geldiğini sayar. `int(c)` her sayıyı yazdırmak için sade bir Python sayısına dönüştürür. Grafikte her değişken, her tohum için bir tane olmak üzere 20 noktadan oluşan bir satır alır; noktalar birbirini örtmesin diye biraz aşağı ve yukarı yayılır.

**Gözlemle ve düşün.** 2.000 komşuyla yirmi tohum aynı sonucu verir; 100 komşuyla ise başı çeken değişken tohumdan tohuma değişir. Bir firma size tek bir hastanın tek bir LIME açıklamasını gösterseydi, açıklamayı okumadan önce hangi iki sayıyı sorardınız?

**İleri düzey (isteğe bağlı).** Çekirdek genişliği $w$, komşuluğun ne kadar uzağa uzandığını belirler. Çok küçük bir genişlikte yalnızca hastanın neredeyse bütün değerlerini koruyan komşular sayılır ve ağırlıklı model düzleşir. Çok büyük bir genişlikte ise her komşu aynı ölçüde sayılır ve LIME ormanı 172 numaralı hastanın yakınında değil, bütün hastalar üzerinde açıklar. `lime(risk_forest, patient, X_train, width=1.0)` ve `width=5.0` değerlerini deneyin ve katsayıları karşılaştırın.

**Alıştırma D.** 172 numaralı hasta için LIME'ı 0,5 çekirdek genişliğiyle, varsayılan 2.000 komşu ve tohumla hesaplayın. Göğüs ağrısı tipinin katsayısı nedir? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("D")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma D", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(lime(risk_forest, patient, X_train, width=0.5)[7]), 3)
teach.check("Alıştırma D", cevap)

## 5. CAMUS'ta düşük ejeksiyon fraksiyonu için bir ağ

*Ders: [Grad-CAM: Bir ağ nereye bakıyor?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#grad-cam-where-does-a-network-look)*

Bir harita değerlendirilmeden önce açıklanacak bir ağ gerekir. 4. haftanın 300 CAMUS hastası, her biri 96 × 96 piksellik bir diyastol sonu ve bir sistol sonu karesiyle ve sol ventrikülün uzman sınırıyla birlikte gelen 600 iki boşluk ve dört boşluk görüntüsü verir. Küçük bir evrişimli ağ, CAMUS'un eğitim hastalarında tek bir görüntüden yüzde 45'in altındaki bir ejeksiyon fraksiyonunu işaretlemeyi öğrenir; doğrulama hastaları dönemi seçer.

**Formüllerden koda.** Tek bir görüntünün iki karesinden oluşan $x$ girdisi: 2 × 96 × 96 boyutlu `echo_x[i]` dizisi. Düşük ejeksiyon fraksiyonunun logiti olan $y^c$ çıktısı: `net(x)`. $M$ uzman sınırı: `outline[i]`; diyastol sonu ya da sistol sonunda kavitenin ve miyokardın içinde doğru (True) değerini alır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 5. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB09_guvenilirlik_ve_aciklanabilir_yapay_zeka.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(5)

In [ ]:
frames_ = camus.frames(camus.patients(), size=96)
info = camus.table()
low_ef = dict(zip(info["patient"], (info["ef"] < 45).astype(int)))
index = {key: i for i, key in enumerate(zip(frames_["patient"], frames_["view"], frames_["phase"]))}
views = sorted(set(zip(frames_["patient"], frames_["view"])))     # 300 hasta x 2 görüntü
echo_x = np.stack([[frames_["images"][index[(p, v, "ED")]], frames_["images"][index[(p, v, "ES")]]] for p, v in views]).astype(np.float32)
labels = [np.isin(frames_["masks"][index[(p, v, phase)]], (1, 2)) for p, v in views for phase in ("ED", "ES")]
outline = np.array(labels).reshape(len(views), 2, 96, 96).any(axis=1)   # ED ya da ES karesinde kavite ya da miyokard
echo_y = np.array([low_ef[p] for p, v in views])
echo_set = np.array([frames_["split"][index[(p, v, "ED")]] for p, v in views])
etr, eva, ete = echo_set == "training", echo_set == "validation", echo_set == "testing"

print(echo_x.shape, "| kümelere göre görüntüler:", {s: int((echo_set == s).sum()) for s in ("training", "validation", "testing")},
      "| düşük ejeksiyon fraksiyonu oranı:", round(echo_y.mean(), 3))
fig, axes = plt.subplots(1, 4, figsize=(9.6, 2.6))
for ax, i in zip(axes, [0, 1, 2, 3]):
    ax.imshow(echo_x[i, 0], cmap="gray")
    ax.contour(outline[i], levels=[0.5], colors="#E8A33D", linewidths=1)
    ax.set_title(f"{views[i][0]} {views[i][1]}, düşük EF {echo_y[i]}", fontsize=8)
    ax.axis("off")
fig.text(0.5, 0.01, "CAMUS, CC BY-NC-SA 4.0", ha="center", fontsize=8)
plt.show()

**Python notu.** `camus.frames`, 4. haftanın hastalarının karelerini ve uzman maskelerini okur ve 96 × 96 piksele getirir. `index` sözlüğü bir hastanın, görüntünün ve evrenin bu dizilerdeki konumunu bulur; `np.stack` de her görüntünün iki karesini 600 × 2 × 96 × 96 boyutlu tek bir dizide birleştirir. `np.isin(mask, (1, 2))` maskenin kaviteyi (1) ya da miyokardı (2) gösterdiği yerlerde doğru değerini verir; `.any(axis=1)` ise diyastol sonu ve sistol sonunu her görüntü için tek bir sınırda birleştirir. `ax.contour(..., levels=[0.5])` sınırın yanlıştan doğruya geçtiği çizgiyi çizer. CAMUS kareleri şeklin altındaki kaynak notuyla gösterilebilir ve hiçbir zaman başkalarıyla paylaşılmaz.

In [ ]:
class EchoNet(nn.Module):
    """Tek bir görüntü için küçük bir evrişimli ağ: Diyastol sonu ve sistol sonu kareleri iki kanal olarak girer; ağ dört bloktan ve tek bir çıktıdan, yani yüzde 45'in altındaki bir ejeksiyon fraksiyonunun logitinden oluşur."""
    def __init__(self):
        super().__init__()
        sizes = [2, 16, 32, 64, 64]
        self.blocks = nn.Sequential(*[nn.Sequential(nn.Conv2d(sizes[i], sizes[i + 1], 3, padding=1), nn.BatchNorm2d(sizes[i + 1]), nn.ReLU(), nn.MaxPool2d(2))
                                      for i in range(4)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 1))

    def forward(self, x):
        return self.head(self.blocks(x)).squeeze(1)


def train_echo(model, X, y, Xv, yv, epochs=30, seed=2026, verbose=True):
    """32 görüntülük gruplarla, öğrenme hızı 0,001 olan Adam; her görüntü en fazla 4 piksel kaydırılır. Doğrulama AUC değeri en iyi olan dönemin ağırlıkları saklanır."""
    torch.manual_seed(seed)
    X, y = torch.as_tensor(X), torch.as_tensor(y, dtype=torch.float32)
    optimiser = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X))
        for i in range(0, len(X), 32):
            b = order[i:i + 32]
            dy, dx = (int(s) for s in torch.randint(-4, 5, (2,)))
            loss = loss_fn(model(torch.roll(X[b], shifts=(dy, dx), dims=(2, 3))), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        a = roc_auc_score(yv, echo_predict(model, Xv))
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
        if verbose and (epoch + 1) % 5 == 0:
            print(f"dönem {epoch + 1}: doğrulama AUC {a:.3f}")
    model.load_state_dict(state)
    return best


@torch.no_grad()
def echo_predict(model, X, batch=64):
    model.eval()
    return torch.cat([torch.sigmoid(model(torch.as_tensor(X[i:i + batch]))) for i in range(0, len(X), batch)]).numpy()

torch.manual_seed(RANDOM_STATE)
net = EchoNet()
print("ağırlıklar:", sum(p.numel() for p in net.parameters()))
train_echo(net, echo_x[etr], echo_y[etr], echo_x[eva], echo_y[eva])
p_echo = echo_predict(net, echo_x)
print("görüntü başına test AUC:", round(roc_auc_score(echo_y[ete], p_echo[ete]), 3))

**Python notu.** `nn.Conv2d(a, b, 3, padding=1)` 3 × 3 piksellik bir pencereyi `a` kanal üzerinde kaydırır ve aynı boyutta `b` kanal verir. `nn.MaxPool2d(2)` yüksekliği ve genişliği yarıya indirir; böylece dört blok 96 × 96 pikseli 6 × 6 boyutuna getirir. `nn.AdaptiveAvgPool2d(1)` her kanalın ortalamasını alır; `squeeze(1)` de (n, 1) biçimindeki çıktıyı n sayıya dönüştürür. `train_echo` içinde `torch.roll` her görüntü grubunu aynı rastgele piksel sayısı kadar kaydırır; böylece ağ kalbin tam yerine dayanmaz. `copy.deepcopy(model.state_dict())` en iyi dönemin ağırlıklarının bir kopyasını saklar. `echo_predict` fonksiyonunun üstündeki `@torch.no_grad()` satırı, tahmin için gerekmeyen gradyanları kapatır.

**Gözlemle ve düşün.** Ağ AUC değerine hiçbir hacim ölçmeden, tek bir görüntünün iki karesinden ulaşır. Kalitesi kötü bir görüntüde ve ejeksiyon fraksiyonu yüzde 45'e yakın bir hastada hangi hataları beklerdiniz?

**Alıştırma E.** Her test hastasının iki görüntüsünün `p_echo` olasılıklarının ortalamasını alın ve 50 test hastası üzerinden AUC değerini hesaplayın. Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("E")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma E", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
patients = np.array([p for p, v in views])
per_patient = pd.DataFrame({"patient": patients[ete], "p": p_echo[ete], "y": echo_y[ete]}).groupby("patient").mean()
cevap = round(float(roc_auc_score(per_patient["y"], per_patient["p"])), 3)
teach.check("Alıştırma E", cevap)

## 6. Uzman sınırına karşı Grad-CAM haritaları

*Ders: [Grad-CAM: Bir ağ nereye bakıyor?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#grad-cam-where-does-a-network-look)*

Grad-CAM son bloğun 64 öznitelik haritasını gradyanlarının ortalamasıyla tartar, toplar ve pozitif kısmı tutar. Her test görüntüsünün haritası uzman sınırıyla karşılaştırılır: Haritanın ağırlığının sınır içindeki payı, en güçlü pikselinin yeri ve ejeksiyon fraksiyonu düşük ve normal görüntüler, doğru ve yanlış tahminler ve üç görüntü kalitesi derecesi için aynı ölçütler.

**Formüllerden koda.** Son bloğun $A^k$ öznitelik haritaları: `saved["maps"]`. $\alpha_k = \frac{1}{Z}\sum_{i,j} \partial y^c / \partial A^k_{ij}$ ağırlıkları: `alpha`. $L = \mathrm{ReLU}(\sum_k \alpha_k A^k)$ haritası: `cam`. Sınırın içindeki pay $\sum M L / \sum L$: `share_inside(cam, mask)`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 6. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB09_guvenilirlik_ve_aciklanabilir_yapay_zeka.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(6)

In [ ]:
def grad_cam(model, X, layer):
    """Bir grup görüntü için çıktının Grad-CAM haritaları: Bir katmanın öznitelik haritaları gradyanlarının ortalamasıyla ağırlıklandırılır, toplanır, 0'ın üstünde kalan kısmı tutulur ve girdinin boyutuna büyütülür."""
    model.eval()
    saved = {}
    hook = layer.register_forward_hook(lambda module, inputs, output: saved.update(maps=output))
    output = model(torch.as_tensor(X))
    hook.remove()
    gradients = torch.autograd.grad(output.sum(), saved["maps"])[0]
    alpha = gradients.mean(dim=(2, 3), keepdim=True)                       # kanal başına bir ağırlık
    cam = torch.relu((alpha * saved["maps"]).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=tuple(X.shape[-2:]), mode="bilinear", align_corners=False)[:, 0]
    return cam.detach().numpy()


def share_inside(cam, mask):
    """Her haritanın ağırlığının sınırın içine düşen payı."""
    return (cam * mask).sum(axis=(1, 2)) / np.maximum(cam.sum(axis=(1, 2)), 1e-12)


def peak_inside(cam, mask):
    """Haritanın en yüksek değerli pikseli sınırın içindeyse 1."""
    return np.array([m.flat[c.argmax()] for c, m in zip(cam, mask)]).astype(int)

cam = grad_cam(net, echo_x[ete], net.blocks[3][2])
mask = outline[ete]
inside, peak = share_inside(cam, mask), peak_inside(cam, mask)
area = mask.reshape(len(mask), -1).mean(axis=1)
quality_table = camus.table().set_index("patient")
quality = np.array([quality_table.loc[p, "quality_2ch" if v == "2CH" else "quality_4ch"] for p, v in views])[ete]
right = (p_echo[ete] >= 0.5) == (echo_y[ete] == 1)
groups = {"hepsi": np.ones(len(cam), bool), "düşük EF": echo_y[ete] == 1, "normal EF": echo_y[ete] == 0,
          "doğru": right, "yanlış": ~right, "Good": quality == "Good", "Medium": quality == "Medium", "Poor": quality == "Poor"}
print(pd.DataFrame({"görüntü": [int(m.sum()) for m in groups.values()],
                    "sınırın içinde": [inside[m].mean() for m in groups.values()],
                    "en güçlü piksel içeride": [peak[m].mean() for m in groups.values()],
                    "sınırın alanı": [area[m].mean() for m in groups.values()]}, index=list(groups)).round(3).to_string())

**Python notu.** `register_forward_hook`, seçilen katman her çalıştığında küçük bir fonksiyonun çağrılmasını sağlar; burada `lambda` katmanın çıktısını, yani öznitelik haritalarını `saved` sözlüğüne kaydeder. `torch.autograd.grad(output.sum(), saved["maps"])` çıktıların bu haritalara göre gradyanını bütün görüntüler için aynı anda verir; çünkü bir görüntünün çıktısı öteki görüntülere bağlı değildir. `mean(dim=(2, 3), keepdim=True)` yükseklik ve genişlik üzerinden ortalama alır ve her kanalı çarpmak için gereken biçimi korur. `F.interpolate(..., mode="bilinear")` 12 × 12 boyutlu haritayı 96 × 96 boyutuna büyütür; `detach()` de sonucu NumPy dizisine dönüşmeden önce gradyanlardan ayırır.

In [ ]:
order = np.argsort(p_echo[ete])
chosen = [order[-1], order[-2], int(np.where(~right & (echo_y[ete] == 1))[0][0]), int(np.where(~right & (echo_y[ete] == 0))[0][0]), order[0],
          int(np.where(quality == "Poor")[0][np.argmax(p_echo[ete][quality == "Poor"])])]
test_views = [views[i] for i in np.where(ete)[0]]
fig, axes = plt.subplots(2, 3, figsize=(9, 7.6))
for ax, k in zip(axes.ravel(), chosen):
    ax.imshow(echo_x[ete][k, 0], cmap="gray")
    ax.imshow(cam[k], cmap="inferno", alpha=0.45)
    ax.contour(mask[k], levels=[0.5], colors="#7FD3F7", linewidths=1)
    ax.set_title(f"{test_views[k][0]} {test_views[k][1]}: p {p_echo[ete][k]:.2f}, düşük EF {echo_y[ete][k]}\niçeride {inside[k]:.2f}", fontsize=8)
    ax.axis("off")
plt.tight_layout(rect=(0, 0.04, 1, 1))
fig.text(0.5, 0.01, "CAMUS, CC BY-NC-SA 4.0", ha="center", fontsize=8)
plt.show()

**Python notu.** `np.argsort` test görüntülerini olasılıklarına göre sıralar; böylece `order[-1]` ve `order[-2]` düşük ejeksiyon fraksiyonu için en emin verilen iki işaret, `order[0]` ise en emin verilen normal görüntü olur. `np.where(~right & (echo_y[ete] == 1))[0][0]` kaçırılan ilk düşük ejeksiyon fraksiyonunu bulur; `~` doğruyu yanlışa, yanlışı doğruya çevirir. Her panel kareyi gri tonlarda, haritayı üstünde `alpha=0.45` ile yüzde 45 saydamlıkta ve sınırı açık mavi bir çizgi olarak çizer.

**Gözlemle ve düşün.** Kaçırılan düşük ejeksiyon fraksiyonuna ve normal bir görüntünün yanlış işaretine bakın. Kalbin üzerinde duran bir harita bir tahmine daha çok güvenmenizi sağlıyor mu? Bir sonucu bir hastaya ya da bir meslektaşınıza açıklamak için kullanmadan önce bir haritanın neyi göstermesi gerekirdi?

**İleri düzey (isteğe bağlı).** Son blok, her biri 8 piksel genişliğinde 12 × 12 hücreli haritalar verir. Üçüncü blok olan `net.blocks[2][2]` ise daha keskin ama karara daha uzak 24 × 24 boyutlu haritalar verir. `grad_cam(net, echo_x[ete], net.blocks[2][2])` haritalarını hesaplayın ve sınırın içindeki paylarını son bloğunkiyle karşılaştırın.

**Alıştırma F.** 100 test görüntüsünün ne kadarında harita, ağırlığının sınır içine düşen payını sınırın görüntüde kapladığı paydan daha büyük tutar? Oranı iki ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("F")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma F", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float((inside > area).mean()), 2)
teach.check("Alıştırma F", cevap)

## 7. Rastgeleleştirilmiş ağlarla sağlamlık testleri

*Ders: [Sağlamlık testleri: Harita modelle mi ilgili?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#sanity-checks-is-the-map-about-the-model)*

Adebayo ve arkadaşlarının model rastgeleleştirme testi, eğitilmiş ağın bir kopyasına önce son blokta ve çıktı katmanında, ardından her katmanda yeni rastgele ağırlıklar verir ve haritaları yeniden hesaplar. Neredeyse hiç değişmeyen haritalar ağın öğrendiklerini değil, görüntüyü anlatır. Tek bir rastgele çekiliş yanıltabileceği için test 1'den 20'ye kadar tohumlarla tekrarlanır.

**Formüllerden koda.** Eğitilmiş $L$ haritası ve rastgeleleştirilmiş bir kopyanın $\tilde{L}$ haritası: `cam` ve `cam_copy`. Bir görüntünün 9.216 pikseli üzerinden Spearman sıra korelasyonu $\rho(L, \tilde{L})$: `spearmanr(a.ravel(), b.ravel())`. Her tohum ve kopya türü için `draws` tablosunda bir satır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 7. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB09_guvenilirlik_ve_aciklanabilir_yapay_zeka.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(7)

In [ ]:
def randomised(model, seed, part):
    """Son blokta ve çıktı katmanında ("top") ya da her katmanda ("all") yeni rastgele ağırlıklar taşıyan bir ağ kopyası."""
    copy_ = copy.deepcopy(model)
    torch.manual_seed(seed)
    layers = list(copy_.blocks[3].modules()) + list(copy_.head.modules()) if part == "top" else [m for m in copy_.modules() if m is not copy_]
    for layer in layers:
        if hasattr(layer, "reset_parameters"):
            layer.reset_parameters()
    return copy_


def rank_correlation(A, B):
    return np.array([spearmanr(a.ravel(), b.ravel()).correlation if a.std() > 0 and b.std() > 0 else np.nan for a, b in zip(A, B)])


rows = []
for part in ("top", "all"):
    for seed in range(1, 21):
        copy_net = randomised(net, seed, part)
        cam_copy = grad_cam(copy_net, echo_x[ete], copy_net.blocks[3][2])
        rho = rank_correlation(cam, cam_copy)
        rows.append({"part": part, "seed": seed, "rho": np.nanmean(rho) if np.isfinite(rho).any() else np.nan,
                     "inside": share_inside(cam_copy, mask).mean(), "empty": int((cam_copy.reshape(len(cam_copy), -1).sum(axis=1) == 0).sum())})
draws = pd.DataFrame(rows)
print(draws.groupby("part")[["rho", "inside"]].agg(["median", "min", "max"]).round(3).to_string())
print("eğitilmiş ağ: içerideki pay", round(inside.mean(), 3), "| sınırın alanı", round(area.mean(), 3))

**Python notu.** `copy.deepcopy(model)` ağın tam bir kopyasını oluşturur; böylece eğitilmiş `net` ağı ağırlıklarını korur. `modules()` ağı ve içindeki bütün katmanları listeler. `reset_parameters()` bir katmana, kurulduğu andaki gibi yeni rastgele ağırlıklar verir; `hasattr` ise katmanın bu yönteme sahip olup olmadığını denetler, çünkü ReLU ve havuzlama katmanlarının ağırlığı yoktur. `a.std() > 0`, bütün pikselleri aynı değeri alan ve sıra korelasyonu tanımlı olmayan bir haritayı dışarıda bırakır; `np.nan` bu değeri eksik olarak işaretler, `np.nanmean` de onu atlar. `groupby("part")[...].agg(["median", "min", "max"])` her kopya türünün 20 çekilişini özetler.

In [ ]:
k = chosen[0]
top_1, all_1 = randomised(net, 1, "top"), randomised(net, 1, "all")
one = echo_x[ete][k:k + 1]
fig, axes = plt.subplots(1, 3, figsize=(8.4, 3))
for ax, (title, m) in zip(axes, [("eğitilmiş", cam[k]),
                                  ("rastgele son blok, tohum 1", grad_cam(top_1, one, top_1.blocks[3][2])[0]),
                                  ("bütün ağırlıklar rastgele, tohum 1", grad_cam(all_1, one, all_1.blocks[3][2])[0])]):
    ax.imshow(echo_x[ete][k, 0], cmap="gray")
    ax.imshow(m, cmap="inferno", alpha=0.45)
    ax.contour(mask[k], levels=[0.5], colors="#7FD3F7", linewidths=1)
    ax.set_title(title, fontsize=8)
    ax.axis("off")
fig.text(0.5, 0.01, "CAMUS, CC BY-NC-SA 4.0", ha="center", fontsize=8)
plt.show()
fig, ax = plt.subplots(figsize=(6.4, 2.6))
for y_, part in enumerate(("top", "all")):
    values = draws.loc[draws["part"] == part, "inside"]
    ax.scatter(values, np.full(len(values), y_) + np.random.default_rng(y_).uniform(-0.12, 0.12, len(values)), color="#1F5F8B", s=16)
ax.axvline(inside.mean(), color="#9B1B1B", label="eğitilmiş ağ")
ax.axvline(area.mean(), color="grey", ls="--", label="sınırın alanı")
ax.set_yticks([0, 1])
ax.set_yticklabels(["rastgele son blok", "bütün ağırlıklar rastgele"])
ax.set_xlabel("haritanın sınır içindeki payı, 20 tohum")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python notu.** İlk şekil tek bir görüntüyü üç kez çizer: Eğitilmiş ağın haritası, son bloğu rastgele olan bir kopyanın haritası ve bütün ağırlıkları rastgele olan bir kopyanın haritası; iki kopya da 1 tohumuyla oluşturulur. `grad_cam` bir grup beklediği için `echo_x[ete][k:k + 1]` tek görüntülük bir grup tutar. Her kopya bir kez oluşturulur; böylece `grad_cam` fonksiyonuna verilen katman aynı kopyaya ait olur. İkinci şekil her türden 20 kopyanın her biri için bir nokta çizer; eğitilmiş ağın payı kırmızı bir çizgiyle, sınırın alanı da kesikli bir çizgiyle gösterilir.

**Gözlemle ve düşün.** Rastgele kopyaların birkaçı haritalarının en az eğitilmiş ağınki kadar büyük bir payını sınırın içine koyar. Öyleyse sınırın içindeki yüksek bir pay neyi kanıtlar ve bir firmadan ek olarak neyi göstermesini isterdiniz?

**Alıştırma G.** Bütün ağırlıkları rastgele olan 20 kopyanın kaçında 100 haritanın yarısından fazlası boştur? `draws` tablosunun `empty` sütununu kullanın.

In [ ]:
veri = alistirma_tablosu("G")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma G", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int(((draws["part"] == "all") & (draws["empty"] > 50)).sum())
teach.check("Alıştırma G", cevap)

## 8. İki hastane arasında veri kayması

*Ders: [Veri kayması: Ne değişti ve bu önemli mi?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#dataset-shift-what-changed-and-does-it-matter)*

8. haftanın göğüs ağrısı ünitesi risk modeli A hastanesinde klinik tablodan, troponinden, ekokardiyogramdan ve triyaj notundan öğrenir ve ardından B hastanesinde kullanılır. B'nin hiçbir sonucu bilinmeden önce, B'nin başvurularını A'nınkilerden ayırmaya çalışan bir sınıflandırıcı girdilerin değişip değişmediğini ve hangi girdilerin değiştiğini gösterir. Hastaneleri birbirinden ayıramayan bir sınıflandırıcının AUC değeri 0,5'e yakın olur.

**Formüllerden koda.** Her başvurunun $x$ girdileri: 39 sütunlu `inputs(A)` ve `inputs(B)`. $h$ hastane etiketi: A için 0, B için 1 olan `hospital`. Beş katlı çapraz doğrulamadan elde edilen alan sınıflandırıcısının AUC değeri: `domain_auc(columns)`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 8. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB09_guvenilirlik_ve_aciklanabilir_yapay_zeka.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(8)

In [ ]:
def fit_logistic(X, y, Cs=(0.01, 0.1, 1.0, 10.0), Xv=None, yv=None):
    """Sütunları standartlaştırır ve bir lojistik regresyon kurar; doğrulama kümesi verilirse doğrulama AUC değeri en büyük olan ters düzenlileştirme gücünü tutar."""
    best = None
    for C in (Cs if Xv is not None else (1.0,)):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)
        auc = roc_auc_score(yv, model.predict_proba(Xv)[:, 1]) if Xv is not None else 0.0
        if best is None or auc > best[1]:
            best = (model, auc)
    return best[0]


def logit(p):
    """Bir olasılığın log-odds değerini (logit) verir."""
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def bootstrap_auc(y, p, n=2000, seed=2026):
    """Hastalardan çekilen n bootstrap örneğinde AUC: n değerden oluşan bir dizi."""
    rng = np.random.default_rng(seed)
    out = []
    while len(out) < n:
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            out.append(roc_auc_score(y[i], p[i]))
    return np.array(out)


def paired_bootstrap(y, p1, p2, n=2000, seed=2026):
    """Aynı hastalardan çekilen n bootstrap örneğinde p1'in AUC değeri eksi p2'nin AUC değeri: n farktan oluşan bir dizi."""
    rng = np.random.default_rng(seed)
    out = []
    while len(out) < n:
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            out.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
    return np.array(out)


def calibration(y, p):
    """Genel kalibrasyon (eğim 1'e sabitlenmişken kesişim) ve kalibrasyon eğimi."""
    z = logit(p)
    a = 0.0
    for _ in range(50):
        q = 1 / (1 + np.exp(-(a + z)))
        a += (y - q).sum() / (q * (1 - q)).sum()
    slope = LogisticRegression(C=1e6, max_iter=5000).fit(z[:, None], y).coef_[0, 0]
    return a, slope


def reliability(y, p, bins=10):
    """Eşit büyüklükte gruplarda ortalama tahmin edilen risk ve gözlenen olay oranı: İki dizi."""
    order = np.argsort(p)
    groups = np.array_split(order, bins)
    return np.array([p[g].mean() for g in groups]), np.array([y[g].mean() for g in groups])


def share_threshold(p, k):
    """Tahmin edilen riski en düşük k hastayı eve gönderen eşik: En düşük k. risk."""
    return float(np.sort(p)[k - 1])


def rule_out_threshold(y, p, sensitivity):
    """Olaylar için duyarlılığı en az verilen düzeyde tutan en büyük eşik. Bu eşikte ya da altında kalan hastalar taburcu edilir. Eşik doğrulama kümesinde seçilir."""
    order = np.argsort(p)
    missed = np.cumsum(y[order])              # her eşikte ya da altında kalan olaylar
    ok = np.where(missed <= (1 - sensitivity) * y.sum())[0]
    return float(p[order][ok.max()]) if len(ok) else 0.0


def wilson(k, n, z=1.96):
    """Bir k/n oranının yüzde 95 Wilson aralığı."""
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (c - h, c + h)


def net_benefit(y, p, thresholds):
    """Her eşiğin üstündeki hastaları tedavi etmenin net yararı: Hasta başına doğru pozitifler eksi eşiğin olabilirliğiyle ağırlıklandırılmış yanlış pozitifler."""
    n = len(y)
    return np.array([((p >= t) & (y == 1)).sum() / n - ((p >= t) & (y == 0)).sum() / n * t / (1 - t) for t in thresholds])

chest = pd.concat([triage.presentations("A"), triage.presentations("B")], ignore_index=True)
A = chest[chest["hospital"] == "A"].reset_index(drop=True)
B = chest[chest["hospital"] == "B"].reset_index(drop=True)
tr, va, te = [(A["set"] == s).to_numpy() for s in ("training", "validation", "test")]
y, yB = A["mace30"].to_numpy(), B["mace30"].to_numpy()

CLINICAL = ["age", "male", "smoker", "diabetes", "hypertension", "hyperlipidaemia", "obesity", "family_history", "known_cad",
            "heart_rate", "sbp", "egfr"]


def clinical(d):
    """Klinik tablo: Yaş, cinsiyet, risk faktörleri, bilinen koroner arter hastalığı, kalp hızı, sistolik basınç ve eGFR."""
    return d.assign(male=(d["sex"] == "M").astype(int))[CLINICAL].astype(float)


def laboratory(d):
    """Kendi testinin üst referans sınırının katları olarak troponin: Gelişteki ve bir saat sonraki değerin logaritması, bir saatteki değişim ve ikinci değerin olup olmadığı."""
    url = d["troponin_url"]
    t0, t1 = d["troponin_0h"] / url, d["troponin_1h"] / url
    return pd.DataFrame({"log_trop_0h": np.log(t0), "log_trop_1h": np.log(t1.fillna(t0)),
                         "delta_1h": (t1 - t0).fillna(0.0), "has_1h": t1.notna().astype(float)})


def echo(d):
    """Yatak başı ekokardiyogram: Yapılıp yapılmadığı, ejeksiyon fraksiyonu (yapılmadıysa 60) ve bölgesel duvar hareket bozukluğu (yapılmadıysa 0)."""
    return pd.DataFrame({"echo_done": d["echo_done"].astype(float), "lvef": d["lvef"].fillna(60.0), "rwma": d["rwma"].fillna(0.0)})

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit(A.loc[tr, "note_tr"])
svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE).fit(tfidf.transform(A.loc[tr, "note_tr"]))


def note(d):
    """Triyaj notu 20 sayı olarak: Kelimelerin ve kelime çiftlerinin TF-IDF değerleri, eğitim notlarında öğrenilen bir tekil değer ayrışımıyla küçültülür."""
    return pd.DataFrame(svd.transform(tfidf.transform(d["note_tr"])), columns=[f"note_{i + 1}" for i in range(20)])

inputs = lambda d: pd.concat([clinical(d), laboratory(d), echo(d), note(d)], axis=1)
XA, XB = inputs(A), inputs(B)
unit_model = fit_logistic(XA[tr], y[tr], Xv=XA[va], yv=y[va])
pA, pB = unit_model.predict_proba(XA)[:, 1], unit_model.predict_proba(XB)[:, 1]

for name, yy, pp in (("A hastanesi, test", y[te], pA[te]), ("B hastanesi", yB, pB)):
    intercept, slope = calibration(yy, pp)
    print(f"{name:22} AUC {roc_auc_score(yy, pp):.3f}   ortalama tahmin {pp.mean():.3f}   gözlenen {yy.mean():.3f}   kesişim {intercept:.2f}   eğim {slope:.2f}")

**Python notu.** Bu hücre 8. haftanın araçlarını ve EKG içermeyen göğüs ağrısı ünitesi modelini tekrarlar: `clinical`, `laboratory`, `echo` ve `note` bir başvuru tablosunu sayılara dönüştürür, `inputs` da bunları 39 sütunda birleştirir. `fit_logistic` ceza gücünü A'nın doğrulama başvurularında seçer. `calibration` kesişimi, yani ortalama tahmin edilen riski olay oranına eşitleyen logit kaymasını ve eğimi döndürür. Döngü aynı anda üç *demet* üzerinde ilerler: `for name, yy, pp in (...)` yazımında her demet üç ada açılır.

In [ ]:
hospital_X = pd.concat([XA, XB], ignore_index=True)
hospital = np.r_[np.zeros(len(XA)), np.ones(len(XB))]          # 0: A hastanesi, 1: B hastanesi
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)


def domain_auc(columns):
    """Verilen sütunlarla kurulan bir lojistik regresyonun B hastanesini A hastanesinden ne kadar iyi ayırdığı; tahminler beş katlı çapraz doğrulamadan gelir."""
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    p = cross_val_predict(model, hospital_X[columns], hospital, cv=folds, method="predict_proba")[:, 1]
    return roc_auc_score(hospital, p)

groups = {"bütün girdiler": list(hospital_X.columns), "klinik tablo": CLINICAL,
          "troponin": ["log_trop_0h", "log_trop_1h", "delta_1h", "has_1h"], "ekokardiyogram": ["echo_done", "lvef", "rwma"],
          "triyaj notu": [f"note_{i + 1}" for i in range(20)]}
for name, columns in groups.items():
    print(f"{name:26} alan sınıflandırıcısının AUC değeri {domain_auc(columns):.3f}")
words = lambda d: d["note_tr"].str.split().str.len()
print(pd.DataFrame({"A": [words(A).mean(), A["age"].mean(), A["diabetes"].mean(), A["egfr"].mean()],
                    "B": [words(B).mean(), B["age"].mean(), B["diabetes"].mean(), B["egfr"].mean()]},
                   index=["nottaki kelime", "yaş", "diyabet", "eGFR"]).round(2))

**Python notu.** `pd.concat([XA, XB], ignore_index=True)` iki hastanenin girdilerini alt alta koyar; `np.r_[np.zeros(len(XA)), np.ones(len(XB))]` de her satır için hastane etiketini yazar. `cross_val_predict` her başvuruya onu görmemiş bir modelden gelen bir olasılık verir; böylece AUC değeri şişmez. `words` adlı lambda her notu `.str.split()` ile boşluklardan böler ve parçaları `.str.len()` ile sayar.

**Gözlemle ve düşün.** Not tek başına iki hastaneyi neredeyse kusursuz biçimde ayırır; klinik tablo ise bunu ancak kısmen yapar. Bir risk modeli için bu farklardan hangisi sizi daha çok kaygılandırırdı: Daha kısa notlar mı, yoksa daha yaşlı, daha çok diyabetli ve eGFR değeri daha düşük hastalar mı? Neden?

**Alıştırma H.** Alan sınıflandırıcısı notun 20 sütunu olmadan hastaneleri ne kadar iyi ayırır? `domain_auc` fonksiyonunu öteki 19 sütunla kullanın ve AUC değerini üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("H")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma H", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(domain_auc([c for c in hospital_X.columns if not c.startswith("note_")])), 3)
teach.check("Alıştırma H", cevap)

## 9. B hastanesinde gözlenen ve beklenen olaylar

*Ders: [Kullanıma alındıktan sonra izlem: Gözlenen ve beklenen olaylar](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#monitoring-after-deployment-observed-against-expected-events)*

Model kullanıma alındıktan sonra hastaların sonuçları birbiri ardına gelir. Gözlenen ve beklenen olaylar arasındaki birikimli fark kalibre bir model için 0'a yakın kalır ve hasta sayısıyla genişleyen bir bandın içinde durur. Eğri bandın dışına çıkıp orada kaldığında model gereğinden az ya da fazla olay bekliyordur; kurul da modelin kesişimini ilk yerel hastalarla günceller.

**Formüllerden koda.** $D_m = \sum_{i \le m} (y_i - p_i)$ farkı: `difference`. $1{,}96\sqrt{\sum_{i \le m} p_i(1 - p_i)}$ bandı: `band`. Kesişimin güncellenmesi, yani logitlerin $a$ kadar kaydırılması: `calibration(yB[:600], pB[:600])[0]`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 9. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB09_guvenilirlik_ve_aciklanabilir_yapay_zeka.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(9)

In [ ]:
def observed_minus_expected(y_seq, p_seq):
    """Başvurular geliş sırasıyla ilerlerken gözlenen olayların beklenen olaylardan farkının birikimli toplamı ve kalibre bir modelin içinde kalacağı 1,96 standart sapmalık bant."""
    difference = np.cumsum(y_seq) - np.cumsum(p_seq)
    band = 1.96 * np.sqrt(np.cumsum(p_seq * (1 - p_seq)))
    return difference, band

difference, band = observed_minus_expected(yB, pB)
first = int(np.where(difference > band)[0][0]) + 1
print("bandın üstündeki ilk başvuru:", first, "| 1.000 başvurudan sonra:",
      round(difference[-1], 1), "±", round(band[-1], 1))
shift_600, _ = calibration(yB[:600], pB[:600])
p_600 = 1 / (1 + np.exp(-(logit(pB) + shift_600)))
difference_600, band_600 = observed_minus_expected(yB[600:], p_600[600:])
print("ilk 600 başvuruyla güncelleme: logit kayması", round(shift_600, 3),
      "| 601-1.000. başvurular:", round(difference_600[-1], 1), "±", round(band_600[-1], 1))
order = np.arange(1, len(yB) + 1)
fig, ax = plt.subplots(figsize=(7.6, 3.2))
ax.fill_between(order, -band, band, color="#1F5F8B", alpha=0.15, label="kalibre bir modelin bandı")
ax.plot(order, difference, color="#9B1B1B", label="A'nın modeli")
ax.plot(order[600:], difference_600, color="#0E7A78", label="ilk 600 ile güncellenmiş")
ax.axvline(first, color="grey", ls="--", lw=0.8)
ax.set_xlabel("B hastanesinde geliş sırası")
ax.set_ylabel("gözlenen − beklenen olay")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python notu.** `np.cumsum` değerleri birbiri ardına toplar; böylece `np.cumsum(y_seq) - np.cumsum(p_seq)` her başvurudan sonraki farkı verir. `np.where(difference > band)[0][0]` bandın üstündeki ilk başvurunun konumudur; 0'dan başlayan konuma 1 eklemek onu 1'den başlayan geliş sırasına dönüştürür. Güncelleme modelin eğimini korur ve her logite aynı sayıyı ekler: `logit(pB) + shift_600` ve $1/(1 + e^{-z})$ fonksiyonu yeni olasılıkları verir. Güncellenmiş eğri 601. başvurudan yeniden başlar; böylece güncellemeyi belirleyen hastalarla değerlendirilmez.

**Gözlemle ve düşün.** B'nin eğrisi başvuruların yaklaşık dörtte birinden sonra bandının dışına çıkar ve bir daha geri dönmez. Bir hastanede bu grafiği kim, hangi sıklıkla görmeli ve ilk alarmın verildiği gün ne yapılmalı?

**Alıştırma I.** Modeli B'nin ilk 600 başvurusu yerine ilk 300 başvurusuyla güncelleyin ve güncellenmiş eğriyi 301. başvurudan itibaren izleyin. Fark ilk kez hangi başvuruda bandın dışına, yani `band` değerinin üstüne ya da `-band` değerinin altına düşer? Geliş sırasını saklayın.

In [ ]:
veri = alistirma_tablosu("I")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma I", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
shift_300, _ = calibration(yB[:300], pB[:300])
p_300 = 1 / (1 + np.exp(-(logit(pB) + shift_300)))
difference_300, band_300 = observed_minus_expected(yB[300:], p_300[300:])
cevap = int(np.where(np.abs(difference_300) > band_300)[0][0]) + 301
teach.check("Alıştırma I", cevap)

## 10. Alt gruplar ve farkında olmama yoluyla adalet

*Ders: [Adalet: Model her hasta için çalışıyor mu?](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/ders.html#fairness-does-the-model-work-for-every-patient)*

En düşük riskteki hastaları eve gönderen bir eşik, A'nın doğrulama başvurularında yüzde 98 duyarlılığı koruyacak biçimde seçilir. Test başvuruları, eşiğin bu sözü kadınlar, erkekler, 65 yaş altındaki ve 65 yaş ve üstündeki hastalar için tutup tutmadığını gösterir. Son hücre Cleveland verisine döner ve cinsiyeti lojistik regresyondan çıkarır; bu yaklaşıma farkında olmama yoluyla adalet denir.

**Formüllerden koda.** $t$ eşiği: `rule_out_threshold(y[va], pA[va], 0.98)`. Bir $g$ grubundaki duyarlılık, $1 - \text{kaçan}_g / \text{olay}_g$: tablonun ilgili sütunu. Beş katlı çapraz doğrulamadan gelen, cinsiyet içermeyen tahminler: `X_cad.drop(columns="sex")` üzerinde `cross_val_predict(...)`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 10. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB09_guvenilirlik_ve_aciklanabilir_yapay_zeka.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-09/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(10)

In [ ]:
threshold = rule_out_threshold(y[va], pA[va], 0.98)
subgroups = {"hepsi": np.ones(len(A), bool), "kadınlar": (A["sex"] == "F").to_numpy(), "erkekler": (A["sex"] == "M").to_numpy(),
             "65 yaş altı": (A["age"] < 65).to_numpy(), "65 yaş ve üstü": (A["age"] >= 65).to_numpy()}
rows = []
for name, m in subgroups.items():
    m = m & te
    home = pA[m] <= threshold
    rows.append({"grup": name, "n": int(m.sum()), "MACE": int(y[m].sum()), "ortalama tahmin": pA[m].mean(),
                 "gözlenen": y[m].mean(), "eve giden": int(home.sum()), "bunlarda MACE": int(y[m][home].sum()),
                 "duyarlılık": 1 - y[m][home].sum() / y[m].sum()})
print("eşik", round(threshold, 4))
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Python notu.** `rule_out_threshold` doğrulama başvurularını riske göre sıralar ve eşikte ya da altında kalan olayların bütün olayların yüzde 2'sini aşmadığı en büyük riski alır. Döngüde `m = m & te` her grubun test başvurularını tutar; `pA[m] <= threshold` da eve gönderilenleri işaretler. `to_string(index=False)` tabloyu satır numaraları olmadan yazdırır.

In [ ]:
folds_cad = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, X_ in (("cinsiyetli", X_cad), ("cinsiyetsiz", X_cad.drop(columns="sex"))):
    p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X_, y_cad, cv=folds_cad, method="predict_proba")[:, 1]
    for sex_name, code_ in (("kadınlar", 0), ("erkekler", 1)):
        m = (X_cad["sex"] == code_).to_numpy()
        yy, pp = y_cad.to_numpy()[m], p[m]
        rows.append({"model": name, "cinsiyet": sex_name, "ortalama tahmin": pp.mean(), "gözlenen": yy.mean(),
                     "duyarlılık": ((pp >= 0.5) & (yy == 1)).sum() / (yy == 1).sum(),
                     "özgüllük": ((pp < 0.5) & (yy == 0)).sum() / (yy == 0).sum(),
                     "yanlış pozitif": int(((pp >= 0.5) & (yy == 0)).sum()), "yanlış negatif": int(((pp < 0.5) & (yy == 1)).sum())})
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Python notu.** Döngü aynı lojistik regresyonu beş katlı çapraz doğrulamayla iki kez kurar: Bir kez on sütunun hepsiyle, bir kez de `X_cad.drop(columns="sex")` ile. `(pp >= 0.5) & (yy == 1)` doğru pozitifleri işaretler; bunları `.sum()` ile toplayıp hastalığı olan hasta sayısına bölmek duyarlılığı verir. İç döngü 303 hastayı `sex` sütununa göre kadınlara ve erkeklere ayırır; model bu sütunu kullanmasa da karşılaştırma için sütun elde kalır.

**Gözlemle ve düşün.** Cinsiyet olmadan model, Cleveland verisindeki kadınlar için bu kadınlardaki hastalık oranından çok daha yüksek bir risk öngörür ve erkeklerde daha çok hastalığı kaçırır. Bir modeli adil kılmak için bir değişkeni modelden çıkarır mıydınız? Bunun yerine neyi, kime bildirirdiniz?

**Alıştırma J.** `pA` risk modelinin kadınların test başvurularındaki AUC değeri nedir? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("J")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma J", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
women = (A["sex"] == "F").to_numpy() & te
cevap = round(float(roc_auc_score(y[women], pA[women])), 3)
teach.check("Alıştırma J", cevap)

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Acil kardiyoloji | 10. bölümdeki eşikte eve gönderilen ve MACE görülen A test başvuruları için MACE olasılığını açıklayın. 39 girdi üzerinde lojistik regresyonun kapalı formunu kullanın ve girdileri klinik tablo, troponin, ekokardiyogram ve not olarak gruplayın. Bu hastaların her birini eşiğin altına hangi grup itmiştir? |
| Girişimsel kardiyoloji | Ormanın Shapley değerlerini tipik anjinası olan ve ağrısı olmayan test hastaları için karşılaştırın. Her grupta tahmini hangi değişkenler belirler ve Cleveland verisinde hastaların anjiyografi için seçilme biçimi bu cevabı nasıl etkiler? |
| Kalp yetersizliği | Eko ağını yalnızca dört boşluk görüntüleriyle eğitin ve AUC değerini ve Grad-CAM haritalarını iki görüntüyle eğitilmiş ağınkilerle karşılaştırın. Dört boşluk ağının haritaları sol ventrikülün üzerine daha sık düşüyor mu? |
| Klinik araştırma | Adebayo ve arkadaşlarının ikinci testini uygulayın: Eko ağını, eğitim görüntüleri arasında rastgele karıştırılmış etiketlerle yeniden eğitin ve haritalarını eğitilmiş ağınkilerle sıra korelasyonu ve sınırın içindeki payla karşılaştırın. Ağ anlamı olmayan etiketlerden ne öğrenir? |
| Kadın ve kalp | Cleveland verisinde test hastaları arasındaki kadınlar ve erkekler için ormandaki cinsiyetin Shapley değerlerini karşılaştırın. Ardından kadınlarda ve erkeklerde göğüs ağrısı tipine göre hastalık oranını inceleyin. Modelin cinsiyetten öğrendiklerinin ne kadarı anjiyografiye kimlerin gönderildiğini yansıtır? |
| Kalite ve güvenlik | B hastanesindeki göğüs ağrısı ünitesi risk modeli için yarım sayfalık bir izlem planı yazın: Neyin, hangi sıklıkla ve kim tarafından grafiğe döküleceğini, alarmı, alarmdan sonraki ilk adımları, güncelleme kuralını ve modelin hangi durumda kullanımdan kaldırılacağını belirtin. |
| Tıp bilişimi | Bu haftanın eko ağı için bir model kartı yazın: Kullanım amacı, veriler ve lisansları, aralıklarıyla ve görüntü kalitesine göre başarım, açıklama yöntemi ve tohumlarla tekrarlanan sağlamlık testi, bilinen sınırlar, izlem planı ve bir iletişim bilgisi. |

## Defterden sonra

[Etkileşimli labı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-09/lab.html) açın ve *Kendini sına* sekmesindeki öz değerlendirmeyi yapın. Ardından *Yansıt ve dışa aktar* sekmesinde yansıtma yazınızı yazın ve öğrenme günlüğünü dışa aktarın. Bu haftanın haftalık projesi 5-7. bölümleri genişletir: Eko ağı aralıklarıyla değerlendirilir, Grad-CAM haritaları görüntü gruplarında uzman sınırlarıyla karşılaştırılır, sağlamlık testi 20 tohumla tekrarlanır ve altı görüntü, haritaların neyi destekleyip neyi destekleyemeyeceğini anlatan yarım sayfalık bir notla kurula sunulur.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>